<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/02.Data_Cleaning_for_LLM_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 2 — Data Cleaning for LLM Training

*LLM Design Patterns* · notebook pendamping, edisi siap Colab GPU

> **Cara membuka notebook ini.** Revisi ini belum dipublikasikan di repositori mana pun, jadi belum ada tautan "Open in Colab". Buka [colab.research.google.com](https://colab.research.google.com), pilih **File → Upload notebook**, lalu unggah berkas `02.Data_Cleaning_for_LLM_Training.COLAB_GPU_READY.ipynb` ini. Setelah terbuka, pilih **Runtime → Change runtime type → T4 GPU**, lalu **Runtime → Run all**.

Ambil dua versi satu kalimat: *The quick brown fox jumps over the lazy dog.* dan *Th3 qu1ck br0wn f0x jumps 0ver th3 l@zy d0g.* Mata kita langsung tahu mana yang kotor. Pertanyaan bab ini lebih sulit: apa artinya "bersih" untuk data yang akan **melatih** model bahasa, dan bagaimana kita tahu sebuah langkah pembersihan benar-benar membantu, bukan sekadar membuat teks tampak rapi?

Buku menjawabnya dalam tujuh bagian, masing-masing dengan skrip Python. Notebook ini menjalankan versi skrip yang sudah diperbaiki, lalu menguji klaimnya di empat eksperimen laboratorium, dengan model yang membaca data itu dan model yang dilatih dengannya.

**Cara membaca angka.** Semua sel diserahkan tanpa output; angka yang muncul setelah *Run all* adalah hasil run Anda. Narasi mengutip dua jenis angka lain, selalu dengan labelnya:

- **run historis**: eksekusi notebook asal di mesin CPU, dengan GPT-2 124M, wikitext-2, seed, dan ukuran sampel yang sama. Di GPU, perplexity biasanya sama sampai beberapa desimal; teks hasil sampling dan model yang dilatih bisa berbeda lebih jauh, jadi yang dibahas adalah arah dan urutan efeknya.
- **validasi lokal**: uji penyusun revisi ini di CPU macOS, dikutip hanya untuk bagian yang memang berjalan di CPU. Ini bukan uji CUDA atau Colab.

Laboratorium juga memakai **kontrol sintetis**: data yang kerusakannya kita tanam sendiri, sehingga labelnya diketahui pasti (*oracle*). Label itu mengukur apakah pemeriksa menangkap kerusakan yang ditanam, bukan penilaian kualitas oleh manusia.

**Di mana GPU dipakai.** Perplexity GPT-2, pembangkitan teks, dan pelatihan model kecil berjalan di GPU, dan notebook berhenti di awal bila GPU tidak berfungsi. Pembersihan teks (spaCy, NLTK, langdetect, TF-IDF, MinHash, pandas) memang algoritma CPU; sel-selnya ditandai `CPU`.

## Fondasi: alur pembersihan data di buku

Figure 2.1 buku menggambarkan alur sebelum pelatihan: pemeriksaan kualitas awal, prapemrosesan, dan deduplikasi; data yang belum memenuhi standar masuk pipeline pembersihan otomatis, lalu divalidasi, dan yang lolos ditandai siap latih. Tujuh bagian bab mengikuti urutan itu.

Untuk setiap skrip kita ajukan dua pertanyaan. Pertama, apakah kodenya berjalan dan melakukan yang dikatakan prosanya? Bila tidak, potongan listing aslinya ditampilkan di kotak singkat, diikuti versi yang berjalan. Kedua, yang lebih penting: apakah "pembersihan" itu membuat data lebih baik untuk model bahasa? Ukurannya bukan tampilan teks, melainkan apa yang terjadi pada model yang membaca atau dilatih dengan data itu.

## Setup

Empat langkah pendek: memasang empat paket yang tidak ada di Colab, mengatur parameter, memeriksa GPU, lalu memuat korpus dan model penilai. Pemeriksaan GPU sengaja diletakkan sebelum pekerjaan berat. Bila GPU tidak ada atau tidak berfungsi, notebook berhenti dengan petunjuk, bukan diam-diam berjalan lambat di CPU.

In [1]:
# Empat paket yang tidak ada di runtime Colab. torch, transformers, datasets, spaCy (+ en_core_web_sm), NLTK,
# scikit-learn, scipy, dan pandas sudah terpasang di Colab. Keempatnya paket baru, jadi runtime tidak perlu di-restart.
%pip install -q langdetect==1.0.9 Unidecode==1.4.0 pyspellchecker==0.9.1 datasketch==2.0.0

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import html
import importlib
import io
import math
import random
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

import datasets
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers

ALLOW_CPU = False         # opt-in eksplisit: True hanya untuk uji jalan tanpa GPU (profil diperkecil, angka tidak sebanding)
SCORER_ID = "gpt2"        # penilai perplexity: GPT-2 124M, pengganti id fiktif "GPT4" di listing buku
SCORE_DTYPE = torch.float32   # perplexity dibandingkan antar-teks: fp32 di semua GPU (lihat catatan di bawah sel gate)
SEED = 42
WORK = Path("ch02_work")  # berkas CSV yang dibuat notebook ini
WORK.mkdir(exist_ok=True)

transformers.logging.set_verbosity_error()
datasets.logging.set_verbosity_error()
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)

torch 2.11.0+cu130 | transformers 5.20.2 | datasets 4.8.5


In [3]:
def cuda_gate(n=2048):
    # Cek fungsional, bukan sekadar is_available(): alokasi, matmul, sinkronisasi, hasil finite.
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA tidak tersedia. Di Colab: Runtime > Change runtime type > T4 GPU, lalu Runtime > Run all. "
            "Untuk uji jalan tanpa GPU, set ALLOW_CPU = True di sel parameter (profil diperkecil).")
    dev = torch.device("cuda")
    x = torch.randn(n, n, device=dev)
    y = (x @ x.T).sum()
    torch.cuda.synchronize()
    if not torch.isfinite(y).item():
        raise RuntimeError("Gate CUDA gagal: hasil matmul tidak finite")
    free, total = torch.cuda.mem_get_info()
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} | compute capability {cap[0]}.{cap[1]} | "
          f"memori bebas {free / 2**30:.1f} dari {total / 2**30:.1f} GiB | bf16 native: {cap >= (8, 0)}")
    del x, y
    torch.cuda.empty_cache()
    return dev


if torch.cuda.is_available() or not ALLOW_CPU:
    DEVICE = cuda_gate()
    PROFILE = "gpu"
else:
    DEVICE = torch.device("cpu")
    PROFILE = "cpu_smoke"
    print("PERINGATAN: ALLOW_CPU = True dan CUDA tidak ada -> profil cpu_smoke. Sampel kecil dan pelatihan dipangkas; "
          "angkanya hanya uji jalan, bukan hasil bab.")

CFG = {
    "gpu": dict(N_NOISE=200, N_GEN=100, N_QA=100, N_DEDUP=500, K_DEDUP=25, TRAIN_STEPS=600, TRAIN_SEEDS=(0, 1),
                N_VAL_SEEDS=5, N_CROSS=None, SCALE_N=(1000, 2000, 4000, 8000)),
    "cpu_smoke": dict(N_NOISE=20, N_GEN=8, N_QA=10, N_DEDUP=100, K_DEDUP=5, TRAIN_STEPS=20, TRAIN_SEEDS=(0,),
                      N_VAL_SEEDS=2, N_CROSS=2000, SCALE_N=(500, 1000)),
}[PROFILE]
print("device:", DEVICE, "| profil:", PROFILE, CFG)

GPU: Tesla T4 | compute capability 7.5 | memori bebas 14.4 dari 14.6 GiB | bf16 native: False
device: cuda | profil: gpu {'N_NOISE': 200, 'N_GEN': 100, 'N_QA': 100, 'N_DEDUP': 500, 'K_DEDUP': 25, 'TRAIN_STEPS': 600, 'TRAIN_SEEDS': (0, 1), 'N_VAL_SEEDS': 5, 'N_CROSS': None, 'SCALE_N': (1000, 2000, 4000, 8000)}


Gate di atas mengalokasikan dua matriks 2048 × 2048 di GPU, mengalikannya, menunggu GPU selesai (`synchronize`), lalu memeriksa hasilnya finite. Bila salah satu langkah gagal, sel berhenti. Satu-satunya jalan ke CPU adalah `ALLOW_CPU = True`, dan itu memilih profil `cpu_smoke` yang jauh lebih kecil.

**Presisi.** Penilai GPT-2 dan model kecil laboratorium berjalan dalam fp32. GPU T4 tidak punya bf16 cepat, dan fp16 tidak diperlukan untuk model sekecil ini. fp32 menjaga perplexity antar-teks tetap sebanding tanpa galat presisi.

Berikutnya korpus kerja bab ini, diunduh dari Hugging Face Hub (CPU dan I/O):

In [4]:
# CPU/I-O: wikitext-2-raw-v1 dari Hugging Face Hub
WT = {s: datasets.load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split=s)["text"] for s in ("train", "validation")}


def is_par(line):
    # Baris paragraf: bukan baris kosong dan bukan judul '= ... ='
    s = line.strip()
    return s != "" and not s.startswith("=")


for split, lines in WT.items():
    n_empty = sum(1 for l in lines if not l.strip())
    n_head = sum(1 for l in lines if l.strip().startswith("="))
    print(f"{split}: {len(lines):,} baris = {n_empty:,} kosong + {n_head:,} judul + {sum(map(is_par, lines)):,} paragraf | "
          f"{sum(len(l.encode()) for l in lines):,} byte")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

train: 36,718 baris = 12,951 kosong + 6,211 judul + 17,556 paragraf | 10,914,845 byte
validation: 3,760 baris = 1,299 kosong + 620 judul + 1,841 paragraf | 1,144,248 byte


**wikitext-2-raw-v1** berisi artikel Wikipedia dalam teks mentah, satu baris per paragraf. Bentuk mentahnya penting untuk bab ini: dari 36.718 baris train, 12.951 kosong dan 6.211 adalah judul bagian (`= ... =`); hanya 17.556 yang paragraf. Baris kosong dan judul sengaja dibiarkan masuk pipeline buku di bagian 6, karena begitulah korpus mentah biasanya datang.

Model penilai dimuat sekali ke GPU dan dipakai di seluruh notebook:

In [5]:
tok = transformers.GPT2TokenizerFast.from_pretrained(SCORER_ID)
lm = transformers.GPT2LMHeadModel.from_pretrained(SCORER_ID).to(device=DEVICE, dtype=SCORE_DTYPE).eval()
assert next(lm.parameters()).device.type == DEVICE.type
CTX = lm.config.n_positions
print(f"{SCORER_ID}: {sum(p.numel() for p in lm.parameters()) / 1e6:.0f} juta parameter | konteks {CTX} token | "
      f"{next(lm.parameters()).dtype} di {next(lm.parameters()).device}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

gpt2: 124 juta parameter | konteks 1024 token | torch.float32 di cuda:0


## 1. Pentingnya data bersih

Klaim buku: kualitas data latih langsung menentukan kinerja dan keandalan LLM. Ilustrasinya membandingkan perplexity kalimat bersih dan versi berderaunya.

**Perplexity** mengukur seberapa "terkejut" model oleh sebuah teks. Model membaca token demi token, dan di setiap posisi memberi peluang pada token berikutnya. Rata-rata minus-log peluang token yang benar disebut *negative log-likelihood* (NLL) per token, dan perplexity adalah eksponennya. Perplexity 10 kira-kira berarti model seragu orang yang memilih di antara 10 token yang sama mungkin, di setiap langkah.

> **Listing buku: contoh yang perlu diperbaiki (bagian 1, langkah 1–2)**
>
> ```python
> pip install torch
> pip install transformers
> ...
>         outputs = model(inputs, labels=inputs["input_ids"])
> ...
> model = GPT4LMHeadModel.from_pretrained("GPT4")
> tokenizer = GPT4Tokenizer.from_pretrained("GPT4")
> ```
>
> Run historis berhenti di tiga tempat. `pip install` adalah perintah shell, sehingga sel Python gagal dengan `SyntaxError` di baris pertama. `GPT4LMHeadModel` dan `GPT4Tokenizer` tidak ada di transformers (`NameError`); buku sendiri, di bagian validasi, menyebut "GPT4" ilustratif dan menyarankan id seperti `gpt2`. Setelah keduanya diganti, `model(inputs, ...)` masih gagal di dalam GPT-2 (`AttributeError`), karena seluruh `BatchEncoding` dikirim sebagai argumen posisi pertama. Versi di bawah memakai sel `%pip` di Setup, model `gpt2`, `model(**inputs, ...)`, dan memindahkan input ke device model.

In [6]:
def calculate_perplexity(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)   # input ikut ke device model (GPU)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])         # ** membongkar input_ids dan attention_mask
    return torch.exp(outputs.loss).item()


clean_text = "The quick brown fox jumps over the lazy dog."
noisy_text = "Th3 qu1ck br0wn f0x jumps 0ver th3 l@zy d0g."
clean_perplexity = calculate_perplexity(lm, tok, clean_text)
noisy_perplexity = calculate_perplexity(lm, tok, noisy_text)
print(f"Clean text perplexity: {clean_perplexity:.2f}")
print(f"Noisy text perplexity: {noisy_perplexity:.2f}")

Clean text perplexity: 162.47
Noisy text perplexity: 587.94


Run historis memberi **162,47** untuk kalimat bersih dan **587,94** untuk versi berderau: model kira-kira 3,6 kali lebih ragu saat membaca versi berderau. Buku menyebut angka yang "mungkin sekitar" 10,25 dan 52,87; angka itu ilustratif dan bergantung pada model, jadi yang bisa dibandingkan hanya arahnya, dan arahnya sama.

Ada satu ketidakcocokan dengan prosa. Buku memperkenalkan skrip ini sebagai pembanding model yang **dilatih** pada data bersih dan berderau, padahal skripnya memakai satu model pra-latih untuk **membaca** dua kalimat. Yang terukur adalah reaksi model terhadap teks berderau, bukan dampaknya pada pelatihan. Eksperimen pelatihan itu dijalankan di UTH-2.

Satu detail yang nanti penting: dengan `labels=input_ids`, transformers menggeser label satu posisi, sehingga token pertama tidak pernah diprediksi. Kalimat bersih 10 token menghasilkan 9 prediksi, dan teks satu token tidak menghasilkan prediksi sama sekali: perplexity-nya `nan`.

### Penilai batch

Laboratorium menilai ribuan teks, jadi kita butuh versi batch dari fungsi yang sama, dengan tiga penjagaan. **Padding bermasker:** teks pendek dipadding di kanan, dan NLL hanya dihitung di posisi token asli. **Batas konteks:** GPT-2 hanya melihat 1.024 token; teks yang lebih panjang dipotong secara eksplisit dan dihitung. **Memori terbatas:** batch dibatasi jumlah token, dan logit untuk 50.257 kata dihitung per teks hanya di posisi yang sah. Tidak ada `try/except`: kehabisan memori GPU atau NLL yang tidak finite menghentikan sel, karena galat seperti itu bukan skor "teks buruk".

Dari NLL itu ada dua ukuran. `ppl_of` memberi perplexity per token (definisi buku). `bpb_of` memberi **bit per byte**, NLL dalam bit dibagi jumlah byte UTF-8, yang tidak bergantung pada cara tokenizer memotong teks.

In [7]:
def score_texts(texts, max_tokens=8192, max_rows=64):
    # NLL GPT-2 per teks, konteks hanya teks itu sendiri (definisi calculate_perplexity buku).
    # Kembali: list (nll_total_nats, n_token_diprediksi, n_byte_token_diprediksi, terpotong)
    enc, cut = [], []
    for t in texts:
        ids = tok(t)["input_ids"]
        cut.append(len(ids) > CTX)
        enc.append(ids[:CTX])
    out = [(float("nan"), 0, 0, c) for c in cut]               # teks < 2 token: tidak ada token yang diprediksi
    order = sorted((i for i in range(len(texts)) if len(enc[i]) >= 2), key=lambda i: len(enc[i]))
    b = 0
    while b < len(order):
        rows = [order[b]]
        while (b + len(rows) < len(order) and len(rows) < max_rows
               and (len(rows) + 1) * len(enc[order[b + len(rows)]]) <= max_tokens):
            rows.append(order[b + len(rows)])
        b += len(rows)
        L = len(enc[rows[-1]])
        ids = torch.full((len(rows), L), tok.eos_token_id, dtype=torch.long)
        att = torch.zeros((len(rows), L), dtype=torch.long)
        for k, i in enumerate(rows):
            ids[k, :len(enc[i])] = torch.tensor(enc[i])
            att[k, :len(enc[i])] = 1
        ids, att = ids.to(DEVICE), att.to(DEVICE)
        with torch.no_grad():
            hidden = lm.transformer(input_ids=ids, attention_mask=att).last_hidden_state
            nll = []
            for k, i in enumerate(rows):
                n = len(enc[i]) - 1                                  # posisi 0..n-1 memprediksi token 1..n
                logits = lm.lm_head(hidden[k, :n]).float()           # n x vocab, bukan batch x L x vocab
                nll.append(F.cross_entropy(logits, ids[k, 1:n + 1], reduction="sum"))
            nll = torch.stack(nll).cpu().tolist()
        for i, s in zip(rows, nll):
            if not math.isfinite(s):
                raise FloatingPointError(f"NLL tidak finite untuk teks #{i}: periksa model/dtype, ini bukan skor kualitas")
            out[i] = (s, len(enc[i]) - 1, len(tok.decode(enc[i][1:]).encode("utf-8")), cut[i])
    return out


def ppl_of(scores):
    # Perplexity per token (definisi buku); NaN untuk teks tanpa token yang diprediksi
    return np.array([math.exp(s / n) if n else float("nan") for s, n, _, _ in scores])


def bpb_of(scores):
    # Bit per byte: NLL dalam bit dibagi jumlah byte UTF-8 yang dicakup token yang diprediksi
    return np.array([s / math.log(2) / nb if nb else float("nan") for s, _, nb, _ in scores])

Sekarang pasangan buku dilihat dari dekat, sekaligus memeriksa bahwa `score_texts` memberi angka yang sama dengan fungsi buku, baik sendiri-sendiri maupun dalam satu batch berpadding:

In [8]:
single = [score_texts([t])[0] for t in (clean_text, noisy_text)]
batched = score_texts([clean_text, noisy_text])               # satu batch: kalimat bersih dipadding ke 23 token
for name, t, (nll, n, nbytes, _), (nll_b, _, _, _) in zip(("clean_text", "noisy_text"), (clean_text, noisy_text), single, batched):
    book = calculate_perplexity(lm, tok, t)
    assert abs(math.exp(nll / n) - book) / book < 1e-3, "penilai batch tidak cocok dengan fungsi buku"
    assert abs(nll_b - nll) / nll < 1e-4, "padding mengubah NLL: masker salah"
    print(f"{name}: {n + 1} token {tok.tokenize(t)}")
    print(f"   NLL total {nll:.2f} nats atas {n} token terprediksi | perplexity {math.exp(nll / n):.2f} "
          f"(fungsi buku {book:.2f}) | {nll / math.log(2) / nbytes:.3f} bit per byte ({nbytes} byte)")

clean_text: 10 token ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.']
   NLL total 45.81 nats atas 9 token terprediksi | perplexity 162.47 (fungsi buku 162.47) | 1.612 bit per byte (41 byte)
noisy_text: 23 token ['Th', '3', 'Ġqu', '1', 'ck', 'Ġbr', '0', 'wn', 'Ġf', '0', 'x', 'Ġjumps', 'Ġ0', 'ver', 'Ġth', '3', 'Ġl', '@', 'zy', 'Ġd', '0', 'g', '.']
   NLL total 140.29 nats atas 22 token terprediksi | perplexity 587.94 (fungsi buku 587.94) | 4.819 bit per byte (42 byte)


**Derau memecah kata menjadi lebih banyak token.** Pada run historis, kalimat bersih 10 token dan versi berderau 23 token, karena setiap angka dan `@` memotong kata menjadi potongan BPE (`Th`, `3`, `Ġqu`, `1`, `ck`, ...). Akibatnya pada ukuran per token:

| run historis | bersih | berderau | rasio |
|---|---|---|---|
| NLL total (nats) | 45,81 | 140,29 | 3,1× |
| token yang diprediksi | 9 | 22 | 2,4× |
| NLL per token (nats) | 5,09 | 6,38 | 1,25× |
| bit per byte | 1,612 | 4,819 | 3,0× |

Total "kejutan" model naik 3,1 kali, tetapi dibagi atas 2,4 kali lebih banyak token, sehingga per token hanya naik 1,25 kali. Bit per byte membagi dengan jumlah byte yang hampir sama (41 dan 42), jadi kenaikan 3 kalinya terlihat utuh. Perplexity per token mencampur dua hal: seberapa aneh sebuah teks, dan berapa token yang dibuat tokenizer darinya. UTH-1 memperlihatkan akibatnya pada 200 kalimat.

## 2. Masalah kualitas umum

Buku mendaftar tujuh masalah: salah eja dan tata bahasa, format tak konsisten, data redundan, konten tak relevan atau berkualitas rendah, kalimat tak lengkap atau terpotong, campur kode, dan informasi pribadi (PII). Untuk mendeteksi sebagiannya, skrip spaCy memeriksa tiga hal: kata salah eja, "skor tata bahasa" dari jumlah kelas kata, dan kalimat yang terlalu pendek.

> **Listing buku: contoh yang perlu diperbaiki (`analyze_text_quality`)**
>
> ```python
>     misspelled = [
>         token.text for token in doc if token._.is_misspelled
>     ]
>     pos_counts = Counter(token.pos_ for token in doc)
>     grammar_score = pos_counts['NOUN'] + pos_counts['VERB']
>     + pos_counts['ADJ'] + pos_counts['ADV']
> ```
>
> Tiga masalah teramati di run historis. Fungsinya tercetak sebagai empat blok berselang prosa: blok lanjutan yang dijalankan sendiri gagal dengan `IndentationError`, sedangkan blok pertama sendirian adalah fungsi sah yang diam-diam mengembalikan `None`. Atribut `token._.is_misspelled` memicu `AttributeError [E046]`, karena spaCy tidak punya pemeriksa ejaan bawaan; `._.` adalah ruang ekstensi yang harus didaftarkan sendiri. Baris kedua `grammar_score` adalah pernyataan terpisah yang dihitung lalu dibuang, jadi skor tercetak hanya NOUN + VERB. Versi di bawah menyambung blok-bloknya, mendaftarkan ekstensi dengan pemeriksa ejaan sungguhan (pyspellchecker), dan memberi kurung pada rumusnya.

In [9]:
# CPU: spaCy dan pyspellchecker
import spacy
from spacy.tokens import Token
from spellchecker import SpellChecker

if not spacy.util.is_package("en_core_web_sm"):        # sudah ada di Colab; unduh hanya bila belum terpasang
    spacy.cli.download("en_core_web_sm")
    importlib.invalidate_caches()
nlp = spacy.load("en_core_web_sm")
spell = SpellChecker()                                    # kamus frekuensi kata bahasa Inggris


def _is_misspelled(token):
    w = token.text
    return w.isalnum() and any(c.isalpha() for c in w) and bool(spell.unknown([w]))


Token.set_extension("is_misspelled", getter=_is_misspelled, force=True)


def analyze_text_quality(text):
    doc = nlp(text)
    misspelled = [token.text for token in doc if token._.is_misspelled]
    pos_counts = Counter(token.pos_ for token in doc)
    grammar_score = (pos_counts["NOUN"] + pos_counts["VERB"]
                     + pos_counts["ADJ"] + pos_counts["ADV"])      # kurung: keempat kelas kata benar-benar dijumlah
    incomplete_sentences = [sent.text for sent in doc.sents if len(sent) < 3]
    return {"misspelled_words": misspelled, "grammar_score": grammar_score,
            "incomplete_sentences": incomplete_sentences}


print("spaCy", spacy.__version__, "| en_core_web_sm", nlp.meta["version"], "| komponen:", nlp.pipe_names)

spaCy 3.8.16 | en_core_web_sm 3.8.0 | komponen: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


Contoh pemakaian buku, ditambah kalimat rubah untuk membandingkan rumus tercetak (NOUN + VERB) dengan rumus yang dimaksud (empat kelas kata):

In [10]:
text = "This iz a smple txt with sum issues. Incomplet"
print(analyze_text_quality(text))
for t in (text, clean_text):
    pos = Counter(w.pos_ for w in nlp(t))
    print(f"{t!r}\n   POS {dict(pos)} | rumus tercetak {pos['NOUN'] + pos['VERB']} | "
          f"empat kelas kata {analyze_text_quality(t)['grammar_score']}")

{'misspelled_words': ['iz', 'smple', 'txt', 'Incomplet'], 'grammar_score': 5, 'incomplete_sentences': ['Incomplet']}
'This iz a smple txt with sum issues. Incomplet'
   POS {'PRON': 1, 'ADP': 2, 'DET': 1, 'NOUN': 5, 'PUNCT': 1} | rumus tercetak 5 | empat kelas kata 5
'The quick brown fox jumps over the lazy dog.'
   POS {'DET': 2, 'ADJ': 3, 'NOUN': 2, 'VERB': 1, 'ADP': 1, 'PUNCT': 1} | rumus tercetak 3 | empat kelas kata 6


Pada run historis, pemeriksa ejaan menandai *iz*, *smple*, *txt*, dan *Incomplet*, tetapi melewatkan *sum* (salah ketik untuk *some*). Pemeriksa kamus tidak melihat salah ketik yang kebetulan membentuk kata lain yang sah. Kalimat tak lengkap yang tertangkap hanya *Incomplet*, karena aturannya `len(sent) < 3` token.

**`grammar_score` menghitung kata isi, bukan tata bahasa.** Untuk kalimat buku, kedua rumus sama-sama 5: lima NOUN, tanpa VERB, ADJ, atau ADV. Untuk kalimat rubah, rumus tercetak memberi 3 dan rumus yang dimaksud 6, karena *quick*, *brown*, dan *lazy* adalah ADJ. Rumus mana pun, skornya naik bersama jumlah kata isi, tidak peduli urutan katanya benar atau tidak.

Ketiga pemeriksaan ini **heuristik**: aturan yang menebak masalah dari ciri permukaan teks. Tebakan heuristik bukan label kebenaran. Seberapa sering tebakannya benar diukur di UTH-3, pada data yang masalahnya kita ketahui pasti.

## 3. Prapemrosesan teks

Buku mendaftar tujuh teknik — huruf kecil, penanganan tanda baca, normalisasi spasi, karakter khusus, tokenisasi, normalisasi angka, dan ekspansi kontraksi — dan menyatakan teknik ini mengurangi derau serta membantu model menggeneralisasi. Bagian ini juga menjelaskan **tokenisasi subword**: kata yang jarang dipecah menjadi potongan yang lebih sering muncul, misalnya *unbelievably* menjadi *un*, *believ*, *ably*. Kita periksa dengan tokenizer BPE GPT-2, dengan dan tanpa spasi di depan kata:

In [11]:
for w in ["unbelievably", "unlikely", "believe", "capably"]:
    print(f"{w!r:16} -> {str(tok.tokenize(w)):32} | {' ' + w!r:17} -> {tok.tokenize(' ' + w)}")

'unbelievably'   -> ['un', 'bel', 'iev', 'ably']     | ' unbelievably'   -> ['Ġunbelievably']
'unlikely'       -> ['un', 'likely']                 | ' unlikely'       -> ['Ġunlikely']
'believe'        -> ['bel', 'ieve']                  | ' believe'        -> ['Ġbelieve']
'capably'        -> ['cap', 'ably']                  | ' capably'        -> ['Ġcap', 'ably']


Tanpa spasi di depan, run historis memecah *unbelievably* menjadi `un`, `bel`, `iev`, `ably`, dekat dengan contoh buku. Dengan spasi di depan, yaitu bentuknya di tengah kalimat, kata yang sama menjadi satu token utuh, `Ġunbelievably` (`Ġ` menandai spasi). BPE menyimpan urutan karakter yang sering muncul di data latihnya sebagai satu token, dan " unbelievably" dengan spasi cukup sering muncul. Contoh buku ilustratif, dan hasil ini sesuai cara kerja yang dijelaskan buku sendiri.

> **Listing buku: contoh yang perlu diperbaiki (unduhan data NLTK)**
>
> ```python
> nltk.download('punkt')
> nltk.download('stopwords')
> ```
>
> Di run historis, `word_tokenize` lalu berhenti dengan `LookupError: Resource 'punkt_tab' not found`. Sejak versi 3.9, NLTK membaca tabel Punkt dari paket `punkt_tab`, jadi `punkt` saja tidak cukup. Sel di bawah mengunduh `punkt_tab`; fungsi `preprocess_text` buku sendiri tidak diubah selain blok-bloknya disambung.

In [12]:
# CPU: NLTK
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

for pkg, resource in (("punkt_tab", "tokenizers/punkt_tab/english"), ("stopwords", "corpora/stopwords")):
    try:
        nltk.data.find(resource)                          # sudah ada: tidak perlu unduh
    except LookupError:
        if not nltk.download(pkg, quiet=True):
            raise RuntimeError(f"unduhan data NLTK '{pkg}' gagal")
print("nltk", nltk.__version__)


def preprocess_text(text):
    # Listing buku, dua blok disambung tanpa perubahan isi
    text = text.lower()                                                            # huruf kecil
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')   # normalisasi Unicode
    text = re.sub(r'[^\w\s]', '', text)                                            # buang tanda baca
    text = ' '.join(text.split())                                                  # normalisasi spasi
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    return ' '.join(tokens)


raw_text = "This is an EXAMPLE of text preprocessing... It's quite useful!"
print(f"Original: {raw_text}")
print(f"Preprocessed: {preprocess_text(raw_text)}")

nltk 3.9.1
Original: This is an EXAMPLE of text preprocessing... It's quite useful!
Preprocessed: example text preprocessing quite useful


Kalimat 10 kata menjadi 5: `example text preprocessing quite useful`. *This*, *is*, *an*, dan *of* dibuang sebagai **stopword**, kata fungsi yang sangat sering seperti *the*, *is*, atau *not*. *It's* hilang dengan cara yang lebih halus: penghapusan tanda baca mengubahnya menjadi *its*, yang juga stopword. Dua teknik yang disebut prosa, normalisasi angka dan ekspansi kontraksi, tidak ada di kodenya.

Untuk melihat apa yang hilang, kita bandingkan prapemrosesan buku dengan **normalisasi yang menjaga makna**: menyatukan penulisan Unicode yang setara (NFC), membuang tag HTML dan memulihkan entitasnya, lalu merapikan spasi, tanpa mengubah huruf, tanda baca, angka, atau kata.

In [13]:
def normalize_safe(text):
    # Normalisasi yang menjaga makna: bentuk Unicode NFC, tag HTML dibuang, entitas dipulihkan, spasi dirapikan
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)
    text = text.replace(" ", " ").replace("​", "")
    return " ".join(text.split())


examples = ["It is not good.", "Prices rose 3.5% in 2023, to $1,250.", "The U.S. team won't play.",
            "She moved to São Paulo and met Zoë.", "これは日本語のテキストです。", "<p>Hello &amp; welcome</p>",
            "Café au lait"]                              # 'é' ditulis sebagai e + aksen gabung (dua code point)
for t in examples:
    print(f"{t!r:40} buku -> {preprocess_text(t)!r:28} aman -> {normalize_safe(t)!r}")
print("code point 'Cafe\\u0301':", len("Café"), "-> NFC", len(unicodedata.normalize("NFC", "Café")),
      "| NFKC mengubah makna: 'x²' ->", repr(unicodedata.normalize("NFKC", "x²")))
sw = set(stopwords.words("english"))
print(f"stopword NLTK bahasa Inggris: {len(sw)} kata, termasuk", [w for w in ["not", "no", "nor", "against", "very"] if w in sw])

'It is not good.'                        buku -> 'good'                       aman -> 'It is not good.'
'Prices rose 3.5% in 2023, to $1,250.'   buku -> 'prices rose 35 2023 1250'   aman -> 'Prices rose 3.5% in 2023, to $1,250.'
"The U.S. team won't play."              buku -> 'us team wont play'          aman -> "The U.S. team won't play."
'She moved to São Paulo and met Zoë.'    buku -> 'moved sao paulo met zoe'    aman -> 'She moved to São Paulo and met Zoë.'
'これは日本語のテキストです。'                         buku -> ''                           aman -> 'これは日本語のテキストです。'
'<p>Hello &amp; welcome</p>'             buku -> 'phello amp welcomep'        aman -> 'Hello & welcome'
'Café au lait'                          buku -> 'cafe au lait'               aman -> 'Café au lait'
code point 'Cafe\u0301': 5 -> NFC 4 | NFKC mengubah makna: 'x²' -> 'x2'
stopword NLTK bahasa Inggris: 198 kata, termasuk ['not', 'no', 'nor', 'against', 'very']


Pada run historis, kolom buku memperlihatkan informasi yang dibuang:

- **Negasi.** *It is not good.* menjadi `good`, karena *not*, *no*, *nor*, dan *against* termasuk 198 stopword NLTK. Maknanya terbalik.
- **Angka.** `3.5%` menjadi `35` dan `$1,250` menjadi `1250`. Titik desimal hilang, jadi nilainya berubah sepuluh kali.
- **Aksara.** NFKD lalu `encode('ascii', 'ignore')` mengubah *São Paulo* menjadi `sao paulo`, dan kalimat Jepang menjadi string kosong karena tidak satu pun karakternya ASCII.
- **Markup.** `<p>Hello &amp; welcome</p>` menjadi `phello amp welcomep`: tag melebur ke dalam kata.

Kolom "aman" mempertahankan semuanya kecuali tag HTML. Membuang tag pun keputusan, bukan kebenaran umum: untuk data kode atau dokumentasi HTML, tag adalah isi. Normalisasi juga tidak otomatis netral. NFC hanya menyatukan penulisan yang setara (`e` + aksen gabung menjadi satu karakter `é`, 5 code point menjadi 4), sedangkan NFKC sudah mengubah `x²` menjadi `x2`.

Untuk klasifikasi kantong-kata, membuang stopword dan tanda baca adalah praktik lama yang wajar. Untuk data latih LLM, yang harus belajar menulis teks lengkap, langkah yang sama menghapus persis hal yang harus dipelajari model. Harganya diukur dengan melatih model di UTH-2.

## 4. Data multibahasa dan campur kode

Untuk data multibahasa, buku menyarankan identifikasi bahasa, normalisasi aksara (misalnya transliterasi), dan prapemrosesan khusus bahasa. Untuk **campur kode**, yaitu satu kalimat yang memakai dua bahasa, langkahnya identifikasi bahasa per token. Skripnya mendeteksi bahasa dengan langdetect dan mentransliterasi dengan unidecode.

> **Listing buku: contoh yang perlu diperbaiki (`handle_multilingual_text`)**
>
> ```python
> from langdetect import detect
> from unidecode import unidecode
> ...
>     try:
>         lang = detect(text)
>     except:
>         lang = 'unknown'
> ```
>
> Di run historis, impor pertama gagal dengan `ModuleNotFoundError`: langdetect dan unidecode tidak ada di Colab, dan buku tidak menyebut pemasangannya (sekarang dipasang di sel `%pip`). Dua masalah lain tidak memunculkan error. langdetect memakai sampling acak, sehingga tanpa `DetectorFactory.seed` labelnya bisa berubah antar-panggilan. `except:` kosong juga menelan galat apa pun, termasuk bug. Versi di bawah memasang seed dan hanya menangkap `LangDetectException`, kondisi yang memang diharapkan untuk teks tanpa huruf.

In [14]:
# CPU: langdetect (profil n-gram karakter + sampling acak) dan unidecode
from langdetect import DetectorFactory, detect, detect_langs
from langdetect.lang_detect_exception import LangDetectException
from unidecode import unidecode

DetectorFactory.seed = 0


def handle_multilingual_text(text):
    try:
        lang = detect(text)
    except LangDetectException:          # teks tanpa fitur bahasa, mis. hanya angka atau tanda baca
        lang = "unknown"
    transliterated_text = unidecode(text)
    tokens = word_tokenize(transliterated_text)
    return {"original": text, "language": lang, "transliterated": transliterated_text, "tokens": tokens}


texts = ["This is English text.", "Dies ist deutscher Text.", "これは日本語のテキストです。",
         "This is mixed language text avec un peu de français.", "12345 !!!"]
for t in texts:
    r = handle_multilingual_text(t)
    print(f"{r['language']:>7} | {r['original']}\n          -> {r['transliterated']!r} | {r['tokens']}")

     en | This is English text.
          -> 'This is English text.' | ['This', 'is', 'English', 'text', '.']
     de | Dies ist deutscher Text.
          -> 'Dies ist deutscher Text.' | ['Dies', 'ist', 'deutscher', 'Text', '.']
     ja | これは日本語のテキストです。
          -> 'korehaRi Ben Yu notekisutodesu. ' | ['korehaRi', 'Ben', 'Yu', 'notekisutodesu', '.']
     fr | This is mixed language text avec un peu de français.
          -> 'This is mixed language text avec un peu de francais.' | ['This', 'is', 'mixed', 'language', 'text', 'avec', 'un', 'peu', 'de', 'francais', '.']
unknown | 12345 !!!
          -> '12345 !!!' | ['12345', '!', '!', '!']


Pada run historis, tiga teks pertama terdeteksi benar: `en`, `de`, `ja`. Kalimat campur kode, lima kata Inggris dan lima kata Prancis, mendapat satu label `fr` tanpa tanda bahwa separuhnya Inggris. Skrip ini mendeteksi satu bahasa per teks, jadi contoh campur kode buku diproses seperti teks satu bahasa. Teks tanpa huruf (`12345 !!!`) memicu `LangDetectException` dan menjadi `unknown`, satu-satunya kasus yang memang perlu ditangkap.

Transliterasi Jepang lebih bermasalah: *これは日本語のテキストです。* menjadi `korehaRi Ben Yu notekisutodesu.` Sel berikut membedah deteksi dan transliterasinya:

In [15]:
mixed = texts[3]
print("detect_langs seluruh kalimat:", detect_langs(mixed))
print("per kata:", [(w, detect(w)) for w in re.findall(r"\w+", mixed)])
DetectorFactory.seed = None                                    # sementara tanpa seed, seperti listing buku
unseeded = Counter(detect("text") for _ in range(100))
DetectorFactory.seed = 0
seeded = Counter(detect("text") for _ in range(100))
print("'text' dideteksi 100 kali tanpa seed:", dict(unseeded), "| dengan seed 0:", dict(seeded))
print("unidecode per karakter:", [(c, unidecode(c)) for c in texts[2]])

detect_langs seluruh kalimat: [fr:0.9999947956631361]
per kata: [('This', 'en'), ('is', 'af'), ('mixed', 'so'), ('language', 'tl'), ('text', 'en'), ('avec', 'sv'), ('un', 'fr'), ('peu', 'fr'), ('de', 'es'), ('français', 'fr')]
'text' dideteksi 100 kali tanpa seed: {'en': 90, 'ca': 10} | dengan seed 0: {'en': 100}
unidecode per karakter: [('こ', 'ko'), ('れ', 're'), ('は', 'ha'), ('日', 'Ri '), ('本', 'Ben '), ('語', 'Yu '), ('の', 'no'), ('テ', 'te'), ('キ', 'ki'), ('ス', 'su'), ('ト', 'to'), ('で', 'de'), ('す', 'su'), ('。', '. ')]


- **Seluruh kalimat.** `detect_langs` memberi `fr` dengan peluang 0,999995 di run historis; detektor kalimat tidak melaporkan campuran.
- **Per kata.** Detektor kalimat yang dipanggil per kata hanya benar untuk sekitar separuhnya: *is* menjadi `af`, *mixed* `so`, *language* `tl`. Satu kata terlalu pendek untuk statistik n-gram karakter; identifikasi per token butuh model yang dilatih untuk itu.
- **Keacakan.** Tanpa seed, kata *text* dideteksi `en` 87 kali dan `ca` 13 kali dari 100 di run historis; dengan seed, hasilnya tetap.
- **Transliterasi.** Kana dibaca per bunyi (`ko`, `re`, `ha`; partikel は sebenarnya dilafalkan *wa*), sedangkan kanji 日本語 (*nihongo*) menjadi `Ri Ben Yu`, bacaan Mandarin. Hasilnya romanisasi yang salah, dan aksara aslinya hilang, padahal untuk data latih multibahasa aksara asli itulah yang perlu dipertahankan.

Pustaka lanjutan yang disarankan buku, Polyglot, terakhir dirilis Juli 2016 (16.7.4). Di run historis detektornya gagal diimpor karena dependensi PyICU yang tidak dideklarasikan, jadi notebook ini tidak memasangnya.

## 5. Deduplikasi

Konten ganda membuat model terlalu sering melihat teks yang sama dan membuang komputasi. Buku membahas empat strategi: **exact match** (teks persis sama), **near-duplicate** (isinya sama, kata-katanya sedikit berbeda), **shingling** (memecah teks menjadi potongan beberapa kata yang tumpang-tindih), dan **LSH** (*locality-sensitive hashing*, cara cepat menemukan pasangan mirip tanpa membandingkan semua pasangan).

Skrip buku memakai **TF-IDF**, vektor bobot kata yang tinggi untuk kata yang jarang di korpus, lalu **kemiripan kosinus** antar-vektor: 1 berarti komposisi katanya identik, 0 berarti tidak berbagi kata. Dokumen yang kemiripannya dengan dokumen sebelumnya melewati ambang 0,9 dibuang. Listing ini berjalan setelah tiga bloknya disambung. Semua sel di bagian ini berjalan di CPU (scikit-learn).

In [16]:
# CPU: scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def deduplicate_corpus(corpus, similarity_threshold=0.9):
    # Listing buku, tiga blok disambung tanpa perubahan isi
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus)
    similarity_matrix = cosine_similarity(tfidf_matrix)
    duplicates = set()
    for i in range(len(corpus)):
        for j in range(i + 1, len(corpus)):
            if similarity_matrix[i, j] > similarity_threshold:
                duplicates.add(j)
    return [doc for i, doc in enumerate(corpus) if i not in duplicates]


corpus = ["The quick brown fox jumps over the lazy dog.",
          "A fast auburn fox leaps above the sleepy canine.",
          "The quick brown fox jumps over the lazy dog.",
          "An entirely different sentence about cats."]
deduplicated = deduplicate_corpus(corpus)
print(f"Original corpus size: {len(corpus)} | Deduplicated corpus size: {len(deduplicated)}")
for doc in deduplicated:
    print(f"- {doc}")

Original corpus size: 4 | Deduplicated corpus size: 3
- The quick brown fox jumps over the lazy dog.
- A fast auburn fox leaps above the sleepy canine.
- An entirely different sentence about cats.


Empat kalimat menjadi tiga: salinan persis kalimat rubah dibuang, parafrasenya (*A fast auburn fox...*) dipertahankan. Untuk melihat alasannya, sel berikut mencetak matriks kemiripan, efek ambang, contoh-contoh lain di prosa buku, dan **Jaccard** shingle tiga kata, yaitu ukuran irisan dibagi gabungan dua himpunan shingle:

In [17]:
np.set_printoptions(precision=3, suppress=True)
S = cosine_similarity(TfidfVectorizer().fit_transform(corpus))
print("kemiripan kosinus korpus contoh:\n", S)
print("ambang -> ukuran hasil:", {t: len(deduplicate_corpus(corpus, t)) for t in [0.0, 0.5, 0.9, 1.0]},
      "| diagonal - 1:", [float(S[i, i] - 1) for i in range(4)])
addresses = ["123 Main St, Anytown, CA 91234", "456 Oak Ave, Somecity, NY 56789", "123 Main St, Anytown, CA 91234"]
print("alamat (exact match):", deduplicate_corpus(addresses))
news = ["The company reported a significant increase in quarterly profits.",
        "Quarterly profits saw a large increase, the company reports."]


def shingles(t, k=3):
    # Shingle kata k=3 seperti contoh buku (huruf kecil, tanda baca dibuang)
    w = re.findall(r"\w+", t.lower())
    return [" ".join(w[i:i + k]) for i in range(len(w) - k + 1)]


a, b = set(shingles(news[0])), set(shingles(news[1]))
print(f"berita (near-duplicate buku): kosinus {cosine_similarity(TfidfVectorizer().fit_transform(news))[0, 1]:.3f} | "
      f"Jaccard shingle k=3 {len(a & b) / len(a | b):.3f} | hasil dedup: {len(deduplicate_corpus(news))} dokumen")
print("shingle k=3 kalimat rubah:", shingles(corpus[0]))

kemiripan kosinus korpus contoh:
 [[1.    0.195 1.    0.   ]
 [0.195 1.    0.195 0.   ]
 [1.    0.195 1.    0.   ]
 [0.    0.    0.    1.   ]]
ambang -> ukuran hasil: {0.0: 2, 0.5: 3, 0.9: 3, 1.0: 4} | diagonal - 1: [0.0, 4.440892098500626e-16, 0.0, 2.220446049250313e-16]
alamat (exact match): ['123 Main St, Anytown, CA 91234', '456 Oak Ave, Somecity, NY 56789']
berita (near-duplicate buku): kosinus 0.458 | Jaccard shingle k=3 0.000 | hasil dedup: 2 dokumen
shingle k=3 kalimat rubah: ['the quick brown', 'quick brown fox', 'brown fox jumps', 'fox jumps over', 'jumps over the', 'over the lazy', 'the lazy dog']


- **Matriks.** Di run historis, salinan persis berkosinus 1 dan parafrase rubah hanya 0,195, karena selain *fox* dan *the* kata-katanya berbeda. TF-IDF membandingkan kata, bukan makna.
- **Ambang 0 dan 1,0 berperilaku aneh.** Fungsi buku memakai `>`. Di ambang 0, setiap pasangan yang berbagi satu kata saja dianggap duplikat, sehingga parafrase ikut terbuang (hasilnya 2 dokumen). Di ambang 1,0, salinan persis hanya terbuang bila galat pembulatan floating point membuat kosinusnya sedikit di atas 1. Run historis memberi 4 dokumen (salinan lolos), dengan diagonal matriks yang menyimpang sampai 4,4e-16 dari 1; validasi lokal dengan scikit-learn 1.9.1 memberi 3 dokumen. Di ambang 1,0, hasilnya ditentukan pembulatan dan versi pustaka, bukan isi teks.
- **Alamat** (exact match) bekerja: alamat ketiga terbuang.
- **Berita**, contoh near-duplicate buku, berkosinus 0,458 dan Jaccard shingle-nya 0: tidak satu shingle pun sama. Kedua kalimat itu parafrase, bukan near-duplicate tekstual. TF-IDF, shingling, maupun LSH di atas shingle itu tidak akan menangkapnya dengan pengaturan bawaan.

## 6. Pipeline pembersihan otomatis

Buku mendaftar tujuh komponen pipeline: ingesti, penilaian kualitas, prapemrosesan, deduplikasi, penyaringan, validasi, dan output. Kelas `DataCleaningPipeline` mengimplementasikan tiga di antaranya: prapemrosesan (huruf kecil, buang tanda baca dan stopword), filter panjang 10–1.000 karakter, dan dedup TF-IDF.

> **Listing buku: contoh yang perlu diperbaiki (`DataCleaningPipeline.clean`)**
>
> ```python
>         df = pd.read_csv(input_file)
>         df['text'] = df['text'].apply(self.preprocess)
> ...
> pipeline = DataCleaningPipeline()
> pipeline.clean('input_data.csv', 'cleaned_data.csv')
> ```
>
> Di run historis, `input_data.csv` tidak pernah dibuat buku (`FileNotFoundError`). Setelah dibuat dari data mentah, `pd.read_csv` membaca baris kosong sebagai `NaN` (float), dan `preprocess` berhenti dengan `AttributeError: 'float' object has no attribute 'lower'`. Daftar nilai kosong bawaan pandas juga memuat `N/A`, `NULL`, dan `null`, sehingga placeholder teks ikut menjadi NaN. Di bawah, berkas input dibuat dari 600 baris pertama wikitext-2 validation (plus kolom `id` untuk melacak baris), dan `read_csv` diberi `keep_default_na=False`.

In [18]:
# CPU/I-O: berkas input yang tidak pernah dibuat buku
raw_lines = WT["validation"][:600]
pd.DataFrame({"id": range(len(raw_lines)), "text": raw_lines}).to_csv(WORK / "input_data.csv", index=False)
print(f"input_data.csv: {len(raw_lines)} baris = {sum(1 for l in raw_lines if not l.strip())} kosong + "
      f"{sum(1 for l in raw_lines if l.strip().startswith('='))} judul + {sum(map(is_par, raw_lines))} paragraf")
na_demo = pd.read_csv(io.StringIO("text\nN/A\nNULL\nnone\n-\n"))["text"]
print("read_csv bawaan mengubah menjadi NaN:", dict(zip(["N/A", "NULL", "none", "-"], na_demo.isna())))

input_data.csv: 600 baris = 207 kosong + 99 judul + 294 paragraf
read_csv bawaan mengubah menjadi NaN: {'N/A': True, 'NULL': True, 'none': False, '-': False}


In [19]:
# CPU: listing buku, empat blok disambung; satu perubahan di clean(): keep_default_na=False
stop_words = set(stopwords.words("english"))


class DataCleaningPipeline:
    def __init__(self, similarity_threshold=0.9, min_length=10, max_length=1000):
        self.similarity_threshold = similarity_threshold
        self.min_length = min_length
        self.max_length = max_length
        self.vectorizer = TfidfVectorizer(stop_words="english")

    def preprocess(self, text):
        text = text.lower()
        text = re.sub(r"[^\w\s]", "", text)
        tokens = [word for word in text.split() if word not in stop_words]
        return " ".join(tokens)

    def filter_by_length(self, df):
        return df[(df["text"].str.len() >= self.min_length) & (df["text"].str.len() <= self.max_length)]

    def deduplicate(self, df):
        tfidf_matrix = self.vectorizer.fit_transform(df["text"])
        similarity_matrix = cosine_similarity(tfidf_matrix)
        duplicates = set()
        for i in range(len(df)):
            for j in range(i + 1, len(df)):
                if similarity_matrix[i, j] > self.similarity_threshold:
                    duplicates.add(j)
        return df.drop(df.index[list(duplicates)])

    def clean(self, input_file, output_file):
        df = pd.read_csv(input_file, keep_default_na=False)    # perubahan: baris kosong dan 'N/A' tetap string
        df["text"] = df["text"].apply(self.preprocess)
        df = self.filter_by_length(df)
        df = self.deduplicate(df)
        df.to_csv(output_file, index=False)
        print(f"Cleaned data saved to {output_file}")


pipe = DataCleaningPipeline()
pipe.clean(WORK / "input_data.csv", WORK / "cleaned_data.csv")
raw = pd.read_csv(WORK / "input_data.csv", keep_default_na=False)
cleaned = pd.read_csv(WORK / "cleaned_data.csv", keep_default_na=False)
after_len = pipe.filter_by_length(raw.assign(text=raw["text"].map(pipe.preprocess)))
heads = raw.set_index("id").loc[cleaned["id"], "text"].str.strip().str.startswith("=")
print(f"{len(raw)} baris -> filter panjang {len(after_len)} -> dedup {len(cleaned)} | judul yang lolos sebagai dokumen: {int(heads.sum())}")
for i in cleaned["id"][:3]:
    print(f"   id {i}: {raw['text'][i].strip()[:90]!r}\n      -> {cleaned.set_index('id')['text'][i][:90]!r}")

Cleaned data saved to ch02_work/cleaned_data.csv
600 baris -> filter panjang 359 -> dedup 356 | judul yang lolos sebagai dokumen: 72
   id 1: '= Homarus gammarus ='
      -> 'homarus gammarus'
   id 3: 'Homarus gammarus , known as the European lobster or common lobster , is a species of clawe'
      -> 'homarus gammarus known european lobster common lobster species clawed lobster eastern atla'
   id 5: '= = Description = ='
      -> 'description'


Pada run historis, 600 baris menjadi 359 setelah filter panjang, lalu 356 setelah dedup; yang dibuang dedup adalah dua baris *Rowin Foula doon !* dan satu judul *Appearances*. Pipeline "berhasil", tetapi lihat keluarannya:

- **Teks asli ditimpa.** Kolom `text` kini huruf kecil tanpa tanda baca dan stopword (`homarus gammarus known european lobster ...`). Teks aslinya tidak disimpan, jadi keluaran ini tidak bisa dipakai melatih model yang harus menulis teks normal.
- **Judul menjadi dokumen.** 72 dari 356 dokumen keluaran adalah judul bagian seperti `homarus gammarus` dan `description`. Filter panjang menghitung karakter *setelah* prapemrosesan, dan judul cukup panjang untuk lolos.
- **Jejak baris hilang.** `to_csv(index=False)` membuang indeks. Tanpa kolom `id` tambahan kita, keluaran tidak bisa dipetakan ke baris asalnya untuk dibandingkan.

Ketiganya masalah desain, bukan sintaks. Versi berikut memisahkan dua hal yang dicampur pipeline buku: **teks yang disimpan** untuk pelatihan, dan **kunci pencocokan** yang hanya dipakai untuk mencari duplikat. Setiap baris yang dibuang dicatat alasannya, dan sebuah `assert` memastikan setiap baris input berakhir tepat di salah satu dari dua tabel: disimpan, atau dibuang dengan alasan.

In [20]:
# CPU: pipeline yang menjaga teks asli dan jejak baris
def clean_with_lineage(df, min_chars=10, max_chars=5000):
    df = df.copy()
    text = df["text"].map(normalize_safe)                                     # normalisasi yang menjaga makna (bagian 3)
    key = text.str.lower().str.replace(r"[^\w\s]", "", regex=True).str.split().str.join(" ")   # hanya untuk mencocokkan
    reason = pd.Series("", index=df.index)
    reason[text == ""] = "kosong"
    reason[(reason == "") & text.str.match(r"^=.*=$")] = "judul bagian (struktur, bukan dokumen)"
    reason[(reason == "") & ((text.str.len() < min_chars) | (text.str.len() > max_chars))] = "panjang di luar batas"
    cand = reason == ""
    first_id = df.loc[cand, "id"].groupby(key[cand]).transform("first")      # kemunculan pertama setiap kunci
    dup = cand & (df["id"] != first_id.reindex(df.index))
    reason[dup] = "duplikat kunci ternormalisasi"
    kept = df.loc[reason == ""].assign(text=text[reason == ""])
    removed = df.loc[reason != "", ["id", "text"]].assign(reason=reason[reason != ""],
                                                          dup_of=first_id.reindex(df.index)[reason != ""])
    assert len(kept) + len(removed) == len(df) and set(kept["id"]).isdisjoint(removed["id"])
    return kept, removed


kept, removed = clean_with_lineage(raw)
print(f"{len(raw)} baris -> {len(kept)} disimpan + {len(removed)} dibuang")
print(removed["reason"].value_counts().to_string())
print("duplikat yang dibuang:")
for _, r in removed[removed["reason"].str.startswith("duplikat")].iterrows():
    print(f"   id {r['id']} {r['text'].strip()!r} = id {int(r['dup_of'])} {raw['text'][int(r['dup_of'])].strip()!r}")
print("contoh teks yang disimpan:", repr(kept["text"].iloc[1][:120]))

600 baris -> 292 disimpan + 308 dibuang
reason
kosong                                    207
judul bagian (struktur, bukan dokumen)     99
duplikat kunci ternormalisasi               2
duplikat yang dibuang:
   id 421 'Rowin Foula doon !' = id 419 'Rowin Foula doon !'
   id 425 'Rowin Foula doon ! "' = id 419 'Rowin Foula doon !'
contoh teks yang disimpan: 'Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilo'


Pada validasi lokal, 600 baris berakhir sebagai 292 disimpan dan 308 dibuang: 207 baris kosong, 99 judul, dan 2 duplikat. Kedua duplikat itu baris *Rowin Foula doon !* yang sama dengan kemunculan pertamanya setelah kunci dinormalisasi; kolom `dup_of` mencatat id baris yang dipertahankan, jadi keputusan itu bisa diperiksa ulang. Teks yang disimpan tetap utuh, dengan huruf kapital, tanda baca, angka, dan stopword, sehingga model masih bisa belajar menulis kalimat lengkap.

Judul dibuang dari tabel ini bukan karena tidak berguna, melainkan karena satu baris judul bukan dokumen. Untuk pelatihan, judul sebaiknya digabung kembali dengan paragraf artikelnya. Kebijakan "simpan kemunculan pertama" juga keputusan: UTH-3 memperlihatkan bahwa dengan kebijakan itu, urutan baris di berkas menentukan versi mana yang bertahan.

## 7. Validasi data

Setelah dibersihkan, buku memvalidasi data dengan statistik dasar, jumlah teks pendek, sampel untuk tinjauan manual, tiga pemeriksaan masalah umum (karakter khusus, angka, huruf kapital semua), dan perplexity GPT-2 pada sampel sebagai ukuran kewajaran teks.

> **Listing buku: contoh yang perlu diperbaiki (`validate_cleaned_data`)**
>
> ```python
>     sample = df.sample(n=min(sample_size, len(df)))
> ...
>     model = GPT4LMHeadModel.from_pretrained('GPT4')
> ...
>             outputs = model(inputs, labels=inputs['input_ids'])
> ...
>         f"{sample_perplexities.mean():.2f}"
> ```
>
> Di run historis, statistiknya tercetak lalu fungsi berhenti di `GPT4LMHeadModel` (`NameError`); `model(inputs, ...)` mengulang masalah bagian 1. Setelah diperbaiki, dua masalah tersisa tanpa error. `df.sample` tanpa seed menarik sampel lain di setiap panggilan, sehingga rata-rata perplexity bergeser antar-run. `.mean()` pandas juga melewati `NaN` tanpa peringatan, jadi teks satu token (tanpa prediksi) diam-diam keluar dari rata-rata. Versi di bawah memakai penilai yang sudah ada di GPU, sampel ber-seed, median di samping rata-rata, serta menghitung terang-terangan teks yang tak terdefinisi atau terpotong.

In [21]:
def validate_cleaned_data(df, sample_size=100, seed=0):
    text = df["text"]
    print(f"Total samples: {len(df)} | Average text length: {text.str.len().mean():.2f} | "
          f"Unique samples: {text.nunique()} | Texts shorter than 10 characters: {int((text.str.len() < 10).sum())}")
    issues = {"special_chars": text.str.contains(r"[^a-zA-Z0-9\s]"), "numbers": text.str.contains(r"\d"),
              "all_caps": text.str.isupper()}
    print("Samples with " + " | ".join(f"{k}: {int(v.sum())}" for k, v in issues.items()))
    sample = df.sample(n=min(sample_size, len(df)), random_state=seed)
    sc = score_texts(sample["text"].tolist())
    ppl = ppl_of(sc)
    print(f"Perplexity sampel {len(sample)} (seed {seed}): rata-rata {np.nanmean(ppl):.2f}, median {np.nanmedian(ppl):.2f} | "
          f"tak terdefinisi (< 2 token): {int(np.isnan(ppl).sum())} | terpotong di {CTX} token: {sum(c for *_, c in sc)}")
    return sample.assign(ppl=ppl)


print("== keluaran pipeline buku ==")
val_book = validate_cleaned_data(cleaned)
print("\n== keluaran pipeline yang menjaga teks asli ==")
val_kept = validate_cleaned_data(kept)

== keluaran pipeline buku ==
Total samples: 356 | Average text length: 272.39 | Unique samples: 356 | Texts shorter than 10 characters: 0
Samples with special_chars: 12 | numbers: 217 | all_caps: 0
Perplexity sampel 100 (seed 0): rata-rata 13433.99, median 1257.08 | tak terdefinisi (< 2 token): 0 | terpotong di 1024 token: 0

== keluaran pipeline yang menjaga teks asli ==
Total samples: 292 | Average text length: 509.48 | Unique samples: 292 | Texts shorter than 10 characters: 0
Samples with special_chars: 287 | numbers: 213 | all_caps: 0
Perplexity sampel 100 (seed 0): rata-rata 189.60, median 55.03 | tak terdefinisi (< 2 token): 0 | terpotong di 1024 token: 0


Angka rata-rata sendirian sulit dibaca. Lebih informatif membandingkan baris yang sama sebelum dan sesudah pipeline buku, lalu melihat seberapa jauh rata-rata bergeser bila sampelnya diganti:

In [22]:
raw_by_id = raw.set_index("id")["text"]
ppl_raw = ppl_of(score_texts(raw_by_id[val_book["id"]].tolist()))
ppl_cln = val_book["ppl"].to_numpy()
ok = ~np.isnan(ppl_raw) & ~np.isnan(ppl_cln)
print(f"{ok.sum()} pasangan baris yang sama: teks asli median {np.median(ppl_raw[ok]):.1f} (rata-rata {ppl_raw[ok].mean():.1f}) | "
      f"sesudah pipeline buku median {np.median(ppl_cln[ok]):.1f} (rata-rata {ppl_cln[ok].mean():.1f})")
print(f"   sesudah > asli di {np.mean(ppl_cln[ok] > ppl_raw[ok]):.0%} baris | rasio median {np.median(ppl_cln[ok] / ppl_raw[ok]):.1f}")
means = [np.nanmean(ppl_of(score_texts(cleaned.sample(n=100, random_state=s)["text"].tolist())))
         for s in range(1, 1 + CFG["N_VAL_SEEDS"])]
print("rata-rata perplexity sampel 100 dengan seed lain:", [round(float(m), 1) for m in means])
one_token = [t for t in cleaned["text"] if len(tok(t)["input_ids"]) < 2]
print(f"teks < 2 token di keluaran buku: {len(one_token)} {one_token[:3]} | pd.Series([10.0, nan]).mean() = "
      f"{pd.Series([10.0, float('nan')]).mean()}")

100 pasangan baris yang sama: teks asli median 70.6 (rata-rata 330.0) | sesudah pipeline buku median 1257.1 (rata-rata 13434.0)
   sesudah > asli di 98% baris | rasio median 18.2
rata-rata perplexity sampel 100 dengan seed lain: [5991.2, 8435.9, 9680.2, 12159.5, 11081.1]
teks < 2 token di keluaran buku: 1 ['description'] | pd.Series([10.0, nan]).mean() = 10.0


**Menurut ukuran buku sendiri, pipeline buku membuat data kurang wajar.** Pada run historis, untuk 100 baris yang sama, perplexity median teks asli 75,4 dan versi yang dibersihkan 1.185,6: naik di 96% baris, dengan rasio median 18. Rata-ratanya (247 lawan 11.738) jauh di atas median, karena segelintir teks berperplexity sangat besar menyeret rata-rata; lima sampel lain memberi rata-rata 3.364–6.469. Median lebih stabil daripada satu angka rata-rata. Teks `description` hanya satu token GPT-2, sehingga perplexity-nya tak terdefinisi dan `.mean()` pandas akan melewatinya diam-diam (`pd.Series([10.0, nan]).mean()` memberi 10,0).

**Pemeriksaan pola mengukur prapemrosesan, bukan kualitas.** Pada keluaran buku, `all_caps` pasti 0 karena semua teks sudah huruf kecil, dan `special_chars` hanya menangkap huruf non-ASCII yang lolos dari `\w`. Pada keluaran yang menjaga teks asli, `special_chars` menandai hampir setiap baris, karena kalimat normal punya tanda baca.

**Perplexity rendah juga bukan bukti kualitas.** Ia mengukur kemiripan dengan yang biasa dilihat model; teks yang diulang-ulang, templat, atau keluaran model lain bisa berperplexity sangat rendah, seperti diperlihatkan laboratorium.

## Laboratorium: empat klaim, diuji dengan angka

Bagian 1–7 menjalankan listing buku. Empat eksperimen berikut menguji klaim yang menopangnya:

- **UTH-1:** apakah perplexity selalu naik pada teks berderau, dan apakah teks buatan mesin berperplexity lebih tinggi daripada teks manusia?
- **UTH-2:** apakah data latih yang kotor, atau yang "dibersihkan" dengan prapemrosesan buku, membuat model lebih buruk? Diukur dengan melatih model kecil.
- **UTH-3:** apa yang sebenarnya ditangkap pemeriksa kualitas buku pada data yang kerusakannya kita tanam sendiri?
- **UTH-4:** apa yang ditangkap dan dilewatkan dedup TF-IDF dibanding hash dan MinHash-LSH, bagaimana mencegah kebocoran duplikat antara data latih dan uji, dan bagaimana biayanya tumbuh?

Sel berikut menyiapkan helper bersama: `detok` merapikan format wikitext mentah (`@-@`, spasi sebelum tanda baca), `sentences` mengambil kalimat 10–30 kata dari paragraf, `leet` membuat derau gaya `noisy_text` buku (*a*→`@`, *e*→`3`, *i*→`1`, *o*→`0`) dengan peluang r per huruf, dan `wilson` menghitung interval kepercayaan 95% untuk sebuah proporsi.

In [23]:
def detok(s):
    s = s.replace(" @-@ ", "-").replace(" @,@ ", ",").replace(" @.@ ", ".")
    s = re.sub(r" ([.,;:!?%')\]])", r"\1", s)
    s = re.sub(r"([(\[$]) ", r"\1", s)
    return " ".join(s.replace(" n't", "n't").split())


def sentences(lines, lo=10, hi=30):
    # Kalimat lo-hi kata dari paragraf wikitext: tanpa tanda kutip/'@'/'=', diawali huruf besar
    out = []
    for line in lines:
        if not is_par(line):
            continue
        for piece in line.strip().split(" . "):
            s = detok(piece.rstrip(" .") + " .")
            if lo <= len(s.split()) <= hi and not set('"@=') & set(s) and s[0].isupper():
                out.append(s)
    return out


LEET = {"a": "@", "e": "3", "i": "1", "o": "0"}
LEET_TR = str.maketrans(LEET)


def leet(s, r, seed):
    g = np.random.default_rng(seed)
    return "".join(LEET[c] if c in LEET and g.random() < r else c for c in s)


def wilson(v, z=1.96):
    v = np.asarray(v, float)
    n, p = len(v), v.mean()
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return f"[{max(0.0, c - h):.3f}; {min(1.0, c + h):.3f}]"


POOL = sentences(WT["validation"])
print(f"{len(POOL):,} kalimat 10-30 kata di validation; contoh: {POOL[0]!r}")

4,961 kalimat 10-30 kata di validation; contoh: 'Homarus gammarus, known as the European lobster or common lobster, is a species of clawed lobster from the eastern Atlantic Ocean, Mediterranean Sea and parts of the Black Sea.'


### UTH-1: Perplexity sebagai sinyal kualitas

Buku menyatakan teks berderau "seharusnya secara konsisten" berperplexity lebih tinggi, dan teks buatan mesin "sering" berperplexity lebih tinggi daripada tulisan manusia. Dua uji:

- **(a) Dosis derau.** Huruf *a/e/i/o* pada 200 kalimat validation diganti dengan peluang r = 0,1; 0,25; 0,5; dan 1. Perplexity per token dan bit per byte dilaporkan berdampingan.
- **(b) Teks mesin.** 100 paragraf dipotong menjadi prefix 32 token. Lanjutan manusia adalah 48 token aslinya; GPT-2 menulis 48 token dari prefix yang sama dengan tiga cara decoding. Semua lanjutan dinilai GPT-2 yang sama, sendiri-sendiri.

In [24]:
# GPU: penilaian GPT-2. Pembuatan derau di CPU.
rng = np.random.default_rng(0)
S1 = [POOL[i] for i in sorted(rng.choice(len(POOL), CFG["N_NOISE"], replace=False))]
sc0 = score_texts(S1)
p0, b0 = ppl_of(sc0), bpb_of(sc0)
rows = [dict(r=0.0, berubah="-", token_median=np.median([n + 1 for _, n, _, _ in sc0]), ppl_median=np.median(p0),
             rasio_ppl=1.0, ppl_naik="-", bpb_median=np.median(b0), rasio_bpb=1.0, bpb_naik="-")]
for r in [0.1, 0.25, 0.5, 1.0]:
    noisy = [leet(s, r, 1000 + i) for i, s in enumerate(S1)]
    changed = np.array([a != b for a, b in zip(noisy, S1)])
    sc = score_texts(noisy)
    p, b = ppl_of(sc), bpb_of(sc)
    rows.append(dict(r=r, berubah=f"{changed.sum()}/{len(S1)}", token_median=np.median([n + 1 for _, n, _, _ in sc]),
                     ppl_median=np.median(p), rasio_ppl=np.median(p / p0), ppl_naik=f"{np.mean(p[changed] > p0[changed]):.3f}",
                     bpb_median=np.median(b), rasio_bpb=np.median(b / b0), bpb_naik=f"{np.mean(b[changed] > b0[changed]):.3f}"))
    if r == 1.0:
        full_noisy, p_full, b_full = noisy, p, b
print(pd.DataFrame(rows).round(3).to_string(index=False))
lower = np.where(p_full < p0)[0]
print(f"r = 1: {len(lower)}/{len(S1)} kalimat berperplexity per token LEBIH RENDAH daripada versi bersihnya")
if len(lower):
    i = lower[0]
    print(f"   {S1[i]!r} (ppl {p0[i]:.1f}, {b0[i]:.3f} bit/byte)\n   -> {full_noisy[i]!r} (ppl {p_full[i]:.1f}, {b_full[i]:.3f} bit/byte)")

   r berubah  token_median  ppl_median  rasio_ppl ppl_naik  bpb_median  rasio_bpb bpb_naik
0.00       -          23.5      60.451      1.000        -       1.195      1.000        -
0.10 191/200          30.0     312.108      5.092    0.995       2.197      1.775    1.000
0.25 199/200          38.0     629.384     10.003    1.000       3.160      2.492    1.000
0.50 200/200          54.0     490.486      7.743    1.000       4.122      3.374    1.000
1.00 200/200          74.0     150.348      2.465    0.910       4.742      3.936    1.000
r = 1: 18/200 kalimat berperplexity per token LEBIH RENDAH daripada versi bersihnya
   'The Penfui detachment attacked Japanese shipping taking part in the invasion of Celebes.' (ppl 256.5, 1.506 bit/byte)
   -> 'Th3 P3nfu1 d3t@chm3nt @tt@ck3d J@p@n3s3 sh1pp1ng t@k1ng p@rt 1n th3 1nv@s10n 0f C3l3b3s.' (ppl 152.6, 5.145 bit/byte)


**Perplexity per token tidak naik sebanding dengan derau.** Pada run historis, rasio median terhadap kalimat bersih adalah 5,09 di r = 0,1, memuncak 10,00 di r = 0,25, lalu *turun* ke 7,74 dan 2,47. Bit per byte naik terus, 1,78; 2,49; 3,37; 3,94 kali, dan di setiap dosis semua kalimat yang berubah lebih tinggi daripada versi bersihnya (kolom `bpb_naik` = 1).

Penyebabnya sama dengan pasangan di bagian 1. Derau memecah kata: median 24 token untuk kalimat bersih, 74 di r = 1. NLL total naik, tetapi dibagi atas tiga kali lebih banyak token. Pada derau penuh, 18 dari 200 kalimat (9%) berperplexity per token *lebih rendah* daripada versi bersihnya, padahal bit per byte-nya naik; contoh run historis turun dari 256,5 ke 152,6 per token, sementara naik dari 1,506 ke 5,145 bit per byte. Klaim "secara konsisten" berlaku untuk pasangan contoh buku, tetapi bukan sifat umum perplexity per token. Untuk membandingkan teks yang tokenisasinya sangat berbeda, gunakan bit per byte.

In [25]:
# GPU: pembangkitan dan penilaian GPT-2
from sklearn.metrics import roc_auc_score

pars = [p for p in (detok(l) for l in WT["validation"] if is_par(l)) if len(tok(p)["input_ids"]) >= 90]
rng = np.random.default_rng(2)
P1 = [pars[i] for i in sorted(rng.choice(len(pars), CFG["N_GEN"], replace=False))]
pre_ids = [tok(p)["input_ids"][:32] for p in P1]
human = [tok.decode(tok(p)["input_ids"][32:80]) for p in P1]


def continue_gpt2(seed, batch=25, **decoding):
    torch.manual_seed(seed)
    out = []
    for b in range(0, len(pre_ids), batch):
        ids = torch.tensor(pre_ids[b:b + batch], device=DEVICE)      # semua prefix tepat 32 token: tanpa padding
        with torch.no_grad():
            gen = lm.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=48, min_new_tokens=48,
                              pad_token_id=tok.eos_token_id, **decoding)
        out += [tok.decode(x[32:]) for x in gen.cpu()]
    return out


GEN = {"manusia (wikitext)": human,
       "GPT-2 greedy": continue_gpt2(0, do_sample=False),
       "GPT-2 top-k 40": continue_gpt2(0, do_sample=True, top_k=40, temperature=1.0),
       "GPT-2 sampling murni": continue_gpt2(0, do_sample=True, top_k=0, temperature=1.0)}
PP = {k: ppl_of(score_texts(v)) for k, v in GEN.items()}
ph = PP["manusia (wikitext)"]
y = np.r_[np.zeros(len(ph)), np.ones(len(ph))]


def repeated(t):
    # Proporsi kata yang mengulang kata sebelumnya di teks yang sama
    w = t.split()
    return 1 - len(set(w)) / max(1, len(w))


rows = []
for k, p in PP.items():
    row = dict(lanjutan=k, ppl_median=np.median(p), p25=np.percentile(p, 25), p75=np.percentile(p, 75),
               kata_berulang=np.median([repeated(t) for t in GEN[k]]))
    if k != "manusia (wikitext)":
        row.update(auc_tinggi_mesin=roc_auc_score(y, np.r_[ph, p]), mesin_gt_manusia=np.mean(p > ph), ci95=wilson(p > ph))
    rows.append(row)
print(f"{len(pars)} paragraf >= 90 token; sampel {len(P1)}")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("prefix paragraf 1:", repr(tok.decode(pre_ids[0])))
for k, v in GEN.items():
    print(f"   {k}: {v[0][:110]!r}")

1168 paragraf >= 90 token; sampel 100
            lanjutan  ppl_median    p25     p75  kata_berulang  auc_tinggi_mesin  mesin_gt_manusia           ci95
  manusia (wikitext)      59.544 41.109  95.382          0.145               NaN               NaN            NaN
        GPT-2 greedy       6.610  4.875   8.832          0.462             0.008              0.00 [0.000; 0.037]
      GPT-2 top-k 40      22.944 16.807  31.664          0.157             0.094              0.07 [0.034; 0.137]
GPT-2 sampling murni      77.198 44.424 121.110          0.093             0.599              0.59 [0.492; 0.681]
prefix paragraf 1: 'Meridian was a small village at the start of the American Civil War in 1861. Its strategic position at the railroad junction led to the Confederate construction of several military'
   manusia (wikitext): ' installations to support the war. During the Battle of Meridian in 1864, General William Tecumseh Sherman led'
   GPT-2 greedy: " bases. The town was also home to th

**Arah klaim terbalik untuk dua dari tiga decoding.** Run historis:

| lanjutan | perplexity median | AUC "tinggi = mesin" | mesin > manusia, berpasangan |
|---|---|---|---|
| manusia | 59,5 | – | – |
| GPT-2 greedy | 6,6 | 0,008 | 0/100 |
| GPT-2 top-k 40 | 23,4 | 0,084 | 6/100 |
| GPT-2 sampling murni | 80,9 | 0,661 | 67/100 |

**AUC** di sini adalah peluang sebuah teks mesin acak berperplexity lebih tinggi daripada teks manusia acak: 0,5 berarti tidak bisa dibedakan, 1 berarti selalu lebih tinggi, 0 berarti selalu lebih rendah. Greedy dan top-k mendekati 0. Tanda teks mesin di sini adalah perplexity yang sangat *rendah*, karena model menulis token yang paling ia harapkan sendiri. Hanya sampling murni, yang sesekali mengambil token berpeluang kecil, searah dengan klaim buku. Kolom kata berulang memperlihatkan ciri lain teks greedy: di run historis hampir separuh katanya mengulang kata sebelumnya (0,462, lawan 0,145 untuk teks manusia).

Ini kondisi termudah untuk deteksi, karena penilai sama dengan pembangkitnya, dan arahnya tetap bergantung pada decoding. Pelajaran UTH-1: perplexity mengukur jarak dari "yang biasa dilihat model" ke dua arah, dan perplexity rendah tidak berarti teks berkualitas.

### UTH-2: Data latih bersih lawan kotor, diuji dengan melatih

Inilah eksperimen yang dijanjikan prosa bagian 1. Model bahasa kecil level byte (arsitektur GPT-2, 2 lapis, dimensi 128, konteks 128 byte, 445.952 parameter) dilatih dari nol di GPU pada lima versi wikitext-2 train, dengan anggaran sama: 600 langkah × 32 blok × 128 byte, dua seed. Kelima versinya: **bersih**; **leet 10%, 30%, dan 100% baris**, yaitu sebagian atau semua baris paragraf, dipilih acak, diberi derau leet penuh; dan **prapemrosesan buku**, setiap baris melalui `preprocess_text` bagian 3.

Setiap model dinilai dalam bit per byte pada validation bersih, leet penuh, dan yang diprapemrosesan. Byte dipakai karena kelima model harus menilai urutan simbol yang sama, sedangkan tokenizer yang dilatih pada salah satu versi data akan menguntungkan versi itu. Kosakata byte hanya 256 nilai, jadi tebakan acak bernilai 8 bit per byte.

Kontrolnya ketat: leet mengganti satu byte dengan satu byte, jadi versi bersih dan leet sama panjang, dan dengan seed sama model melihat blok di posisi yang sama. Beda datanya hanya derau. Persiapan data berjalan di CPU:

In [26]:
# CPU: menyiapkan lima versi data latih dan tiga versi validation sebagai byte
T, BATCH = 128, 32


def corrupt(lines, p, seed):
    # Derau leet penuh pada sebagian p baris paragraf, dipilih acak
    g = np.random.default_rng(seed)
    return [l.translate(LEET_TR) if is_par(l) and g.random() < p else l for l in lines]


def book_pre(lines):
    return [preprocess_text(l) + "\n" if l.strip() else l for l in lines]


def to_bytes(lines):
    return torch.frombuffer(bytearray("".join(lines).encode("utf-8")), dtype=torch.uint8)


t0 = time.time()
TRAIN = {"bersih": to_bytes(WT["train"]),
         "leet 10% baris": to_bytes(corrupt(WT["train"], 0.1, 1)),
         "leet 30% baris": to_bytes(corrupt(WT["train"], 0.3, 1)),
         "leet 100% baris": to_bytes(corrupt(WT["train"], 1.0, 1)),
         "prapemrosesan buku": to_bytes(book_pre(WT["train"]))}
VAL = {k: to_bytes(v)[:1024 * T].view(1024, T) for k, v in
       {"bersih": WT["validation"], "leet 100%": corrupt(WT["validation"], 1.0, 2),
        "prapemrosesan buku": book_pre(WT["validation"])}.items()}
LEETS = ["leet 10% baris", "leet 30% baris", "leet 100% baris"]
assert all(len(TRAIN[k]) == len(TRAIN["bersih"]) for k in LEETS)        # substitusi 1 byte -> 1 byte
print("byte data latih:", {k: f"{len(v):,}" for k, v in TRAIN.items()}, f"| persiapan {time.time() - t0:.0f} s")
print("byte yang berbeda dari versi bersih:", {k: f"{int((TRAIN[k] != TRAIN['bersih']).sum()):,}" for k in LEETS})
print("validation: 1.024 blok x 128 byte per versi; awal blok kedua:",
      {k: bytes(v[1, :40].tolist()).decode("utf-8", "replace") for k, v in VAL.items()})

byte data latih: {'bersih': '10,914,845', 'leet 10% baris': '10,914,845', 'leet 30% baris': '10,914,845', 'leet 100% baris': '10,914,845', 'prapemrosesan buku': '7,483,400'} | persiapan 14 s
byte yang berbeda dari versi bersih: {'leet 10% baris': '293,653', 'leet 30% baris': '862,433', 'leet 100% baris': '2,821,101'}
validation: 1.024 blok x 128 byte per versi; awal blok kedua: {'bersih': ' the eastern Atlantic Ocean , Mediterran', 'leet 100%': ' th3 3@st3rn Atl@nt1c Oc3@n , M3d1t3rr@n', 'prapemrosesan buku': 'ean sea parts black sea closely related '}


Model dan loop latihnya. `causal_nll` menuliskan pergeseran kausal secara eksplisit: logit di posisi *t* memprediksi byte *t*+1, jadi logit terakhir dan byte pertama setiap blok tidak dipakai. Sel loop memeriksanya dulu terhadap loss bawaan transformers. Data latih, model, dan batch berada di device yang sama; loss yang tidak finite menghentikan pelatihan, bukan dicatat sebagai skor.

In [27]:
# GPU: model byte-level, loss kausal, evaluasi bit per byte, loop latih
def make_byte_lm(seed):
    torch.manual_seed(seed)                                          # inisialisasi sama untuk setiap lengan ber-seed sama
    cfg = transformers.GPT2Config(vocab_size=256, n_positions=T, n_embd=128, n_layer=2, n_head=4,
                                  resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0, bos_token_id=0, eos_token_id=0)
    return transformers.GPT2LMHeadModel(cfg).to(DEVICE)


def causal_nll(model, x):
    # Rata-rata NLL (nats) memprediksi byte t+1 dari byte 1..t di dalam blok x [B, T]
    logits = model(input_ids=x).logits[:, :-1]
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), x[:, 1:].reshape(-1))


@torch.no_grad()
def bpb_model(model, X, batch=64):
    model.eval()
    total, n = 0.0, 0
    for i in range(0, len(X), batch):
        x = X[i:i + batch].to(DEVICE).long()
        total += causal_nll(model, x).item() * x[:, 1:].numel()
        n += x[:, 1:].numel()
    return total / n / math.log(2)


def train_byte_lm(data, seed, steps):
    model = make_byte_lm(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 50) * 0.5 * (1 + math.cos(math.pi * s / steps)))
    data_dev = data.to(DEVICE)
    offsets = torch.arange(T, device=DEVICE)
    g = torch.Generator().manual_seed(seed)                          # generator CPU: posisi blok sama di lengan sepanjang sama
    model.train()
    for step in range(steps):
        ix = torch.randint(0, len(data) - T, (BATCH,), generator=g).to(DEVICE)
        x = data_dev[ix[:, None] + offsets].long()
        loss = causal_nll(model, x)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"loss tidak finite di langkah {step}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        opt.zero_grad(set_to_none=True)
    return model.eval()

In [28]:
# GPU: 5 lengan x seed, lalu bit per byte pada tiga versi validation
x_chk = VAL["bersih"][:4].to(DEVICE).long()
m_chk = make_byte_lm(0)
with torch.no_grad():
    assert torch.allclose(causal_nll(m_chk, x_chk), m_chk(input_ids=x_chk, labels=x_chk).loss.float(), atol=1e-4)
assert next(m_chk.parameters()).device.type == x_chk.device.type == DEVICE.type
del m_chk, x_chk
print(f"pergeseran kausal sama dengan loss bawaan transformers | {CFG['TRAIN_STEPS']} langkah per run di {DEVICE}")

R2, MODELS = {}, {}
t0 = time.time()
for arm, data in TRAIN.items():
    for seed in CFG["TRAIN_SEEDS"]:
        t1 = time.time()
        model = train_byte_lm(data, seed, CFG["TRAIN_STEPS"])
        R2[arm, seed] = {k: bpb_model(model, X) for k, X in VAL.items()}
        if seed == CFG["TRAIN_SEEDS"][0]:
            MODELS[arm] = model
        print(f"{arm:<20} seed {seed} | {time.time() - t1:5.1f} s | bit/byte " +
              " | ".join(f"val {k} {v:.3f}" for k, v in R2[arm, seed].items()), flush=True)
print(f"parameter per model: {sum(p.numel() for p in model.parameters()):,} | total {time.time() - t0:.0f} s")

pergeseran kausal sama dengan loss bawaan transformers | 600 langkah per run di cuda
bersih               seed 0 |   6.5 s | bit/byte val bersih 2.872 | val leet 100% 7.206 | val prapemrosesan buku 3.428
bersih               seed 1 |   6.3 s | bit/byte val bersih 2.848 | val leet 100% 7.474 | val prapemrosesan buku 3.408
leet 10% baris       seed 0 |   6.3 s | bit/byte val bersih 2.934 | val leet 100% 3.071 | val prapemrosesan buku 3.461
leet 10% baris       seed 1 |   6.3 s | bit/byte val bersih 2.910 | val leet 100% 3.067 | val prapemrosesan buku 3.450
leet 30% baris       seed 0 |   6.4 s | bit/byte val bersih 2.976 | val leet 100% 3.009 | val prapemrosesan buku 3.482
leet 30% baris       seed 1 |   6.4 s | bit/byte val bersih 2.984 | val leet 100% 3.022 | val prapemrosesan buku 3.503
leet 100% baris      seed 0 |   6.4 s | bit/byte val bersih 4.152 | val leet 100% 2.914 | val prapemrosesan buku 4.477
leet 100% baris      seed 1 |   6.4 s | bit/byte val bersih 4.012 | val leet 100% 

Pada run historis (CPU, konfigurasi yang sama), angkanya dalam bit per byte untuk seed 0 / seed 1:

| data latih | val bersih | val leet penuh | val diprapemrosesan |
|---|---|---|---|
| bersih | 2,876 / 2,848 | 7,187 / 7,451 | 3,429 / 3,408 |
| leet 10% baris | 2,940 / 2,911 | 3,074 / 3,069 | 3,466 / 3,451 |
| leet 30% baris | 2,979 / 2,982 | 3,013 / 3,020 | 3,485 / 3,503 |
| leet 100% baris | 4,166 / 4,034 | 2,901 / 2,916 | 4,491 / 4,250 |
| prapemrosesan buku | 4,556 / 4,509 | 8,458 / 8,364 | 2,975 / 2,967 |

Baca kolom pertama dulu, karena teks bersih adalah yang kita ingin model tulis. Makin banyak derau di data latih, makin buruk model pada teks bersih. Model prapemrosesan buku lebih buruk lagi, padahal setiap byte yang dilihatnya "bersih" menurut pipeline buku. Selisih terhadap model bersih ber-seed sama dihitung di sel berikut.

In [29]:
# GPU: selisih terhadap model bersih ber-seed sama, dan teks greedy 100 byte dari prompt yang sama
rows = []
for arm in TRAIN:
    if arm == "bersih":
        continue
    rows.append({"data latih": arm, **{f"selisih val {k}": " / ".join(f"{R2[arm, s][k] - R2['bersih', s][k]:+.3f}"
                                                                         for s in CFG["TRAIN_SEEDS"]) for k in VAL}})
print(pd.DataFrame(rows).to_string(index=False))
prompt = torch.tensor([list(b"The game was released in ")], device=DEVICE)
for arm, m in MODELS.items():
    with torch.no_grad():
        o = m.generate(input_ids=prompt, attention_mask=torch.ones_like(prompt), max_new_tokens=100, do_sample=False, pad_token_id=0)
    print(f"{arm:<20} {bytes(o[0].cpu().tolist()).decode('utf-8', 'replace')!r}")

        data latih selisih val bersih selisih val leet 100% selisih val prapemrosesan buku
    leet 10% baris    +0.062 / +0.061       -4.134 / -4.407                +0.033 / +0.042
    leet 30% baris    +0.103 / +0.136       -4.196 / -4.452                +0.054 / +0.095
   leet 100% baris    +1.280 / +1.164       -4.291 / -4.557                +1.049 / +0.836
prapemrosesan buku    +1.684 / +1.664       +1.252 / +0.890                -0.453 / -0.446
bersih               'The game was released in the seare and the sear and the sear and the sear and the sear and the sear and the sear and the the '
leet 10% baris       'The game was released in the sear and the sear and the sear and the sear and the sear and the sear and the sear and the the t'
leet 30% baris       'The game was released in the sear and the sead the sear the sead the sear a the sto the sear and the sear the sear the sear t'
leet 100% baris      'The game was released in = = = = = = = = = = = = \n = = = = = = = = = = = = 

Selisih run historis terhadap model bersih ber-seed sama (seed 0 / seed 1, bit per byte; positif berarti lebih buruk):

- **Derau di data latih merusak model pada teks bersih, makin banyak makin rusak:** +0,063 / +0,064 untuk 10% baris, +0,103 / +0,134 untuk 30%, dan +1,289 / +1,186 untuk semua baris paragraf. Selisih antar-seed model bersih sendiri 0,028, jadi efek 10% baris pun di atas variasi seed.
- **Prapemrosesan buku merusak lebih dari derau di semua paragraf:** +1,679 / +1,661. Model ini tak pernah melihat huruf kapital, tanda baca, titik desimal, atau stopword yang harus diprediksinya di teks sungguhan. Ia hanya unggul pada teks yang sudah diprapemrosesan (−0,454 / −0,442).
- **Teks yang ditulis model** belum koheren pada 2,8–4,6 bit per byte. Model leet 100% di run historis menulis `= = = =`, penanda judul wikitext, karena judul adalah satu-satunya baris tanpa derau di data latihnya.

Ini model 446 ribu parameter dengan 2,46 juta byte latihan per run: yang didukung adalah arah dan urutan efek data latih, bukan besarnya pada LLM.

### UTH-3: Pemeriksa kualitas buku pada kontrol sintetis

Buku sendiri, di bagian QA, menyarankan dataset sintetis dengan masalah yang diketahui untuk mengukur pipeline. Kita buat **kontrol sintetis**: 100 kalimat validation (sampel lain dari UTH-1), masing-masing dirusak dengan sepuluh cara — salah ketik, leet, terpotong, kata diacak, diulang tiga kali, huruf kapital semua, dibungkus HTML, placeholder pendek (`N/A`, `NULL`, `TBD`, ...), ditambah PII (email dan nomor telepon fiktif), dan ditambah kalimat boilerplate. Label setiap teks diketahui karena kita yang membuat kerusakannya. Label itu *oracle* untuk kerusakan yang ditanam, bukan penilaian kualitas oleh manusia: kalimat "bersih" pun bisa saja punya masalah yang tidak kita tanam.

Setiap pemeriksa buku dijalankan pada 1.100 teks: tiga keluaran `analyze_text_quality` (bagian 2), tiga pemeriksaan pola dan teks di bawah 10 karakter (bagian 7), serta perplexity GPT-2. `grammar_score` dilaporkan dalam dua rumus: tercetak (NOUN + VERB) dan yang dimaksud (empat kelas kata).

In [30]:
# CPU: membuat kontrol sintetis dan menjalankan pemeriksa aturan; GPU: perplexity
from scipy import stats

rng = np.random.default_rng(1)
B3 = [POOL[i] for i in sorted(rng.choice(len(POOL), CFG["N_QA"], replace=False))]


def typo(s, seed):
    # Salah ketik pada ~20% kata alfabetis >= 4 huruf: tukar, hapus, atau ganti satu huruf
    g = random.Random(seed)
    w = s.split()
    idx = [i for i, x in enumerate(w) if re.fullmatch(r"\W*[A-Za-z]{4,}\W*", x)]
    for i in g.sample(idx, min(len(idx), max(1, round(0.2 * len(idx))))):
        a, c, b = re.match(r"^(\W*)([A-Za-z]+)(\W*)$", w[i]).groups()
        k = g.randrange(1, len(c) - 1)
        op = g.choice(["tukar", "hapus", "ganti"])
        if op == "tukar":
            c = c[:k] + c[k + 1] + c[k] + c[k + 2:]
        elif op == "hapus":
            c = c[:k] + c[k + 1:]
        else:
            c = c[:k] + g.choice("abcdefghijklmnopqrstuvwxyz") + c[k + 1:]
        w[i] = a + c + b
    return " ".join(w)


def trunc(s, seed):
    # Potong di 40-60% kata, tanpa titik
    w = s.rstrip(" .").split()
    return " ".join(w[:max(3, round(len(w) * random.Random(seed).uniform(0.4, 0.6)))])


def shuf(s, seed):
    # Kata yang sama, urutan acak
    w = s.rstrip(" .").split()
    random.Random(seed).shuffle(w)
    return " ".join(w) + "."


PH = ["N/A", "n/a", "NULL", "null", "-", "TBD", "none", "...", "[deleted]", "?"]
NAMES = ["john.smith", "maria.garcia", "wei.chen", "a.kumar", "l.dubois"]
rows = []
for i, s in enumerate(B3):
    rows += [("bersih", i, s), ("salah_ketik", i, typo(s, i)), ("leet", i, s.translate(LEET_TR)),
             ("terpotong", i, trunc(s, i)), ("kata_diacak", i, shuf(s, i)), ("ulang_3x", i, " ".join([s] * 3)),
             ("kapital", i, s.upper()), ("html", i, f'<div class="post"><p>{s}</p></div>'), ("pendek", i, PH[i % 10]),
             ("pii", i, s + f" Contact {NAMES[i % 5]}@example.com or 555-01{i % 100:02d}."),
             ("boilerplate", i, s + " Click here to subscribe to our newsletter!")]
D3 = pd.DataFrame(rows, columns=["issue", "base", "text"])
Q = [analyze_text_quality(t) for t in D3["text"]]
D3["eja"] = [len(q["misspelled_words"]) > 0 for q in Q]
D3["tak_lengkap"] = [len(q["incomplete_sentences"]) > 0 for q in Q]
D3["grammar"] = [q["grammar_score"] for q in Q]
D3["grammar_tercetak"] = [(lambda c: c["NOUN"] + c["VERB"])(Counter(w.pos_ for w in nlp(t))) for t in D3["text"]]
D3["special_chars"] = D3["text"].str.contains(r"[^a-zA-Z0-9\s]")
D3["numbers"] = D3["text"].str.contains(r"\d")
D3["all_caps"] = D3["text"].str.isupper()
D3["pendek<10"] = D3["text"].str.len() < 10
D3["ppl"] = ppl_of(score_texts(D3["text"].tolist()))                      # GPU

FLAGS = ["eja", "tak_lengkap", "special_chars", "numbers", "all_caps", "pendek<10"]
clean3 = D3[D3.issue == "bersih"]
table = []
for k, g in D3.groupby("issue", sort=False):
    row = {"masalah": k, **{f: g[f].mean() for f in FLAGS}, "ppl_NaN": int(g.ppl.isna().sum()), "ppl_median": g.ppl.median()}
    if k != "bersih":
        sub = pd.concat([clean3, g])
        yy = (sub.issue != "bersih").astype(int)
        ok = sub.ppl.notna()
        row["AUC_ppl"] = roc_auc_score(yy[ok], sub.ppl[ok]) if yy[ok].nunique() == 2 else float("nan")
        row["AUC_gram_tercetak"] = roc_auc_score(yy, -sub.grammar_tercetak)
        row["AUC_gram_4kelas"] = roc_auc_score(yy, -sub.grammar)
    table.append(row)
print("Bendera = proporsi teks yang ditandai; baris 'bersih' = laju alarm palsu. AUC: bermasalah lawan bersih "
      "(ppl tinggi = bermasalah, grammar rendah = bermasalah).")
print(pd.DataFrame(table).round(3).to_string(index=False))
same = np.mean(D3[D3.issue == "kata_diacak"].grammar_tercetak.to_numpy() == clean3.grammar_tercetak.to_numpy())
print(f"grammar_score tercetak sama persis untuk kalimat dan versi berkata diacak: {same:.2f} | Spearman grammar tercetak "
      f"lawan jumlah kata (bersih): {stats.spearmanr(clean3.grammar_tercetak, clean3.text.str.split().str.len()).statistic:.3f}")
fp = Counter(w for q, k in zip(Q, D3.issue) if k == "bersih" for w in q["misspelled_words"])
print(f"token 'salah eja' di kalimat bersih: {sum(fp.values())}; terbanyak {fp.most_common(6)}")

Bendera = proporsi teks yang ditandai; baris 'bersih' = laju alarm palsu. AUC: bermasalah lawan bersih (ppl tinggi = bermasalah, grammar rendah = bermasalah).
    masalah  eja  tak_lengkap  special_chars  numbers  all_caps  pendek<10  ppl_NaN  ppl_median  AUC_ppl  AUC_gram_tercetak  AUC_gram_4kelas
     bersih 0.52         0.01           1.00     0.49       0.0        0.0        0      69.664      NaN                NaN              NaN
salah_ketik 1.00         0.01           1.00     0.49       0.0        0.0        0     275.973    0.892              0.492            0.502
       leet 1.00         0.06           1.00     1.00       0.0        0.0        0     153.455    0.823              0.759            0.823
  terpotong 0.33         0.01           0.76     0.31       0.0        0.0        0     118.748    0.645              0.884            0.888
kata_diacak 0.52         0.03           1.00     0.49       0.0        0.0        0    2432.434    0.999              0.472            0

Baris `bersih` adalah laju alarm palsu: seberapa sering pemeriksa menandai kalimat yang tidak kita rusak. Pada run historis, setiap pemeriksa aturan gagal dengan caranya sendiri:

- **Ejaan** menangkap salah ketik, leet, dan HTML (1,00), tetapi juga 0,52 kalimat bersih, kebanyakan karena nama diri (*Asahi*, *SpongeBob*, *Kyawswa*).
- **Kalimat tak lengkap** (`len(sent) < 3`) menangkap hanya 0,01 kalimat terpotong, karena kalimat yang terpotong di tengah tetap tiga token atau lebih. Yang tertangkap hanya fragmen, seperti 0,70 placeholder.
- **`special_chars`** menandai semua kalimat bersih (setiap kalimat punya titik), dan **`numbers`** menandai 0,49 di antaranya.
- **`grammar_score` tercetak** sama persis untuk kalimat dan versi berkata diacak di 0,49 pasangan (AUC 0,490, setara tebakan acak), berkorelasi dengan panjang kalimat (Spearman 0,508), dan menilai kalimat yang diulang tiga kali *lebih* gramatikal. Rumus empat kelas kata menghitung kata isi yang sama; bandingkan kedua kolom AUC-nya di output Anda.

**Perplexity** adalah satu-satunya sinyal untuk kata diacak (AUC 0,999) dan cukup kuat untuk salah ketik dan leet (0,892 dan 0,823). Namun kalimat yang diulang tiga kali bermedian 5,3, jauh di *bawah* kalimat bersih (69,7), dengan AUC 0,000. Buku menulis bahwa perplexity sangat tinggi bisa menandai urutan berulang; arahnya terbalik. HTML, huruf kapital, dan boilerplate juga lebih rendah daripada kalimat bersih (AUC 0,31–0,38), karena tag dan frasa langganan mudah ditebak. Tidak satu pemeriksa pun dibuat untuk PII.

Setiap pemeriksa adalah heuristik dengan alarm palsunya sendiri, dan perplexity bukan skor kualitas universal. Keduanya perlu diukur pada data berlabel sebelum dipakai membuang data.

In [31]:
# CPU: pipeline buku pada kontrol sintetis, dengan urutan baris asli dan terbalik
def run_pipeline(df, name):
    df[["issue", "base", "text"]].to_csv(WORK / f"{name}.csv", index=False)
    DataCleaningPipeline().clean(WORK / f"{name}.csv", WORK / f"{name}_clean.csv")
    return pd.read_csv(WORK / f"{name}_clean.csv", keep_default_na=False)


out_f = run_pipeline(D3, "uth3_urut")
out_r = run_pipeline(D3.iloc[::-1], "uth3_terbalik")
flt3 = DataCleaningPipeline().filter_by_length(D3[["issue", "text"]].assign(text=D3["text"].map(pipe.preprocess)))
n = CFG["N_QA"]
table = [{"masalah": k, "lolos filter panjang": f"{int((flt3.issue == k).sum())}/{n}",
          "tersisa (urutan asli)": f"{int((out_f.issue == k).sum())}/{n}",
          "tersisa (urutan terbalik)": f"{int((out_r.issue == k).sum())}/{n}"} for k in D3.issue.unique()]
print(pd.DataFrame(table).to_string(index=False))
print(f"total tersisa: {len(out_f)} (urutan asli) dan {len(out_r)} (terbalik) dari {len(D3)}")
for k in ["html", "leet", "pii"]:
    r = out_f[out_f.issue == k]
    if len(r):
        print(f"   keluaran {k}: {r.text.iloc[0][:100]!r}")

Cleaned data saved to ch02_work/uth3_urut_clean.csv
Cleaned data saved to ch02_work/uth3_terbalik_clean.csv
    masalah lolos filter panjang tersisa (urutan asli) tersisa (urutan terbalik)
     bersih              100/100               100/100                     0/100
salah_ketik              100/100                98/100                    98/100
       leet              100/100               100/100                   100/100
  terpotong               99/100                99/100                    99/100
kata_diacak              100/100                 0/100                     0/100
   ulang_3x              100/100                 0/100                     0/100
    kapital              100/100                 0/100                     1/100
       html              100/100               100/100                   100/100
     pendek                0/100                 0/100                     0/100
        pii              100/100                52/100                    97/100
b

Pada run historis, pipeline buku menyisakan 550 dari 1.100 baris, dan pilihannya tidak mengikuti kualitas:

- **Yang dibuang:** kapital, kata diacak, dan diulang tiga kali (0/100 tersisa), karena setelah huruf kecil, buang stopword, dan TF-IDF ketiganya identik atau hampir identik dengan kalimat bersihnya. TF-IDF tidak melihat urutan kata, dan kalimat yang diulang tiga kali searah dengan aslinya setelah normalisasi panjang vektor.
- **Yang lolos:** leet 100/100, terpotong 99/100, HTML 100/100. HTML lolos sebagai kata baru hasil peleburan tag (`div classpostphis appointment ...`). PII lolos sebagai `johnsmithexamplecom 5550100`: masih terbaca manusia, tetapi tidak lagi cocok dengan pola email untuk disaring.
- **Urutan baris menentukan pemenang.** Dengan urutan terbalik, kalimat bersih tersisa 0/100 dan versi boilerplate 100/100. Dedup menyimpan anggota pertama setiap kelompok mirip, apa pun isinya.

Di dunia nyata kita tidak punya label kontrol sintetis ini. Karena itu kebijakan "mana yang disimpan" perlu aturan eksplisit (misalnya sumber paling tepercaya atau versi terpanjang), bukan kebetulan urutan berkas.

### UTH-4: Deduplikasi pada korpus berlabel

Kontrol sintetis kedua: 500 paragraf unik wikitext-2 train (60 kata pertama setiap paragraf ≥ 60 kata), ditambah 150 near-duplicate dari enam jenis, 25 per jenis, yang diletakkan sesudah aslinya: salinan persis, beda format (huruf kecil, tanpa koma, spasi ganda), 5%, 15%, dan 30% kata diganti kata acak, serta paragraf yang ditambahi tiga kalimat boilerplate. Karena kita tahu varian mana berasal dari paragraf mana, setiap paragraf asli yang dibuang adalah alarm palsu, dan setiap varian yang lolos adalah duplikat yang terlewat.

Metodenya: hash md5 (persis dan setelah dinormalisasi), fungsi TF-IDF buku di enam ambang, dan **MinHash-LSH**. MinHash meringkas himpunan shingle sebuah dokumen menjadi 128 angka sehingga Jaccard dua dokumen bisa diperkirakan cepat; LSH mengelompokkan ringkasan itu ke dalam "ember" sehingga hanya dokumen yang jatuh di ember sama yang dibandingkan. Dokumen seember disebut **kandidat**, belum tentu duplikat. Tabel menampilkan dua cara memakainya: membuang setiap dokumen yang punya kandidat, dan membuang hanya bila Jaccard sebenarnya dengan kandidat itu lolos ambang. Semua sel UTH-4 berjalan di CPU.

In [32]:
# CPU: md5, TF-IDF buku, MinHash-LSH (datasketch) pada korpus berlabel
import hashlib

from datasketch import MinHash, MinHashLSH

N4, K4 = CFG["N_DEDUP"], CFG["K_DEDUP"]
P60 = sorted({" ".join(detok(l).split()[:60]) for l in WT["train"] if is_par(l) and len(detok(l).split()) >= 60})
rng4 = random.Random(0)
base4 = rng4.sample(P60, N4)
vocab4 = sorted({w for p in base4 for w in p.split()})


def edit(p, frac, seed):
    g = random.Random(seed)
    w = p.split()
    for i in g.sample(range(len(w)), round(frac * len(w))):
        w[i] = g.choice(vocab4)
    return " ".join(w)


BOIL = " Subscribe to our newsletter for more stories like this. Share this article with your friends. All rights reserved."
VTYPES = {"persis": lambda p, k: p,
          "format": lambda p, k: "  ".join(re.sub(r"[,;:]", "", p).lower().split(" ")),
          "edit 5%": lambda p, k: edit(p, 0.05, k), "edit 15%": lambda p, k: edit(p, 0.15, k),
          "edit 30%": lambda p, k: edit(p, 0.30, k), "boilerplate": lambda p, k: p + BOIL}
src4 = rng4.sample(range(N4), K4 * len(VTYPES))
lab4 = [(list(VTYPES)[k // K4], i) for k, i in enumerate(src4)]          # (jenis varian, indeks paragraf asal)
C4 = base4 + [VTYPES[t](base4[i], k) for k, (t, i) in enumerate(lab4)]
vtype = {N4 + k: t for k, (t, _) in enumerate(lab4)}


def report(name, removed):
    rec = {t: sum(1 for r in removed if r >= N4 and vtype[r] == t) / K4 for t in VTYPES}
    return {"metode": name, "dibuang": len(removed), "alarm_palsu": sum(1 for r in removed if r < N4), **rec}


def hash_dedup(corpus, f):
    seen, removed = set(), set()
    for i, d in enumerate(corpus):
        h = hashlib.md5(f(d).encode()).hexdigest()
        if h in seen:
            removed.add(i)
        seen.add(h)
    return removed


def tfidf_dedup(corpus, threshold):
    # Replika fungsi buku yang mengembalikan indeks yang dibuang
    M = cosine_similarity(TfidfVectorizer().fit_transform(corpus))
    return {j for i in range(len(corpus)) for j in range(i + 1, len(corpus)) if M[i, j] > threshold}, M


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0


def minhash_of(sh, num_perm=128):
    m = MinHash(num_perm=num_perm, seed=1)
    m.update_batch([s.encode() for s in sh])
    return m


def lsh_dedup(corpus, threshold, verify):
    lsh = MinHashLSH(threshold=threshold, num_perm=128)
    sh = [set(shingles(d)) for d in corpus]
    removed, n_cand, n_false = set(), 0, 0
    for i, s in enumerate(sh):
        m = minhash_of(s)
        cands = lsh.query(m)
        true = [c for c in cands if jaccard(s, sh[int(c)]) >= threshold]
        n_cand += len(cands)
        n_false += len(cands) - len(true)
        hit = true if verify else cands
        if hit:
            removed.add(i)
        lsh.insert(str(i), m)
    return removed, n_cand, n_false, (lsh.b, lsh.r)


rows = [report("md5 teks persis", hash_dedup(C4, lambda d: d)),
        report("md5 dinormalisasi", hash_dedup(C4, lambda d: " ".join(re.sub(r"[^\w\s]", "", d.lower()).split())))]
for th in [0.0, 0.5, 0.8, 0.9, 0.95, 1.0]:
    dup, M4 = tfidf_dedup(C4, th)
    assert deduplicate_corpus(C4, th) == [d for i, d in enumerate(C4) if i not in dup]   # sama dengan fungsi buku
    rows.append(report(f"TF-IDF buku > {th}", dup))
LSH_INFO = {}
for th in [0.5, 0.8]:
    for verify in (False, True):
        rem, n_cand, n_false, br = lsh_dedup(C4, th, verify)
        LSH_INFO[th] = (n_cand, n_false, br)
        rows.append(report(f"MinHash-LSH J {th}" + (" + verifikasi" if verify else " (kandidat saja)"), rem))
print(f"{len(P60):,} paragraf unik >= 60 kata di train; korpus {len(C4)} dokumen")
print(pd.DataFrame(rows).round(2).to_string(index=False))
for th, (n_cand, n_false, (b, r)) in LSH_INFO.items():
    curve = {s: round(1 - (1 - s ** r) ** b, 3) for s in (0.3, 0.5, 0.7, 0.8, 0.9)}
    print(f"LSH J {th}: b = {b} ember x r = {r} baris | kandidat {n_cand}, di bawah ambang setelah dicek {n_false} | "
          f"P(jadi kandidat | Jaccard s) = {curve}")
for t in VTYPES:
    pairs = [(i, N4 + k) for k, (tt, i) in enumerate(lab4) if tt == t]
    cs = [M4[i, j] for i, j in pairs]
    js = [jaccard(set(shingles(C4[i])), set(shingles(C4[j]))) for i, j in pairs]
    print(f"   {t:<12} kosinus TF-IDF median {np.median(cs):.3f} (min {min(cs):.3f}) | Jaccard shingle median {np.median(js):.3f}")
ex = np.array([M4[i, N4 + k] for k, (t, i) in enumerate(lab4) if t == "persis"]) - 1.0
B4 = M4[:N4, :N4].copy()
np.fill_diagonal(B4, 0)
print(f"salinan persis dengan kosinus terhitung > 1,0: {int((ex > 0).sum())}/{K4} | kemiripan tertinggi antar-paragraf asli: {B4.max():.3f}")

12,320 paragraf unik >= 60 kata di train; korpus 650 dokumen
                           metode  dibuang  alarm_palsu  persis  format  edit 5%  edit 15%  edit 30%  boilerplate
                  md5 teks persis       25            0    1.00    0.00     0.00      0.00       0.0         0.00
                md5 dinormalisasi       50            0    1.00    1.00     0.00      0.00       0.0         0.00
                TF-IDF buku > 0.0      649          499    1.00    1.00     1.00      1.00       1.0         1.00
                TF-IDF buku > 0.5      150            0    1.00    1.00     1.00      1.00       1.0         1.00
                TF-IDF buku > 0.8      122            0    1.00    1.00     1.00      0.88       0.0         1.00
                TF-IDF buku > 0.9      100            0    1.00    1.00     1.00      0.00       0.0         1.00
               TF-IDF buku > 0.95       62            0    1.00    1.00     0.40      0.00       0.0         0.08
                TF-IDF buku

Pada run historis, kemiripan tertinggi antar-paragraf asli hanya 0,392, jadi setiap paragraf asli yang dibuang memang alarm palsu. Baris md5 dan TF-IDF sama di validasi lokal, kecuali ambang 1,0 yang bergantung pada pembulatan.

- **Ambang bawaan 0,9** menangkap salinan persis, beda format, edit 5%, dan boilerplate, tetapi tidak satu pun edit 15% (median kosinus 0,841) atau 30%. Ambang 0,5 menangkap semuanya tanpa alarm palsu di korpus ini, sedangkan ambang 0 membuang 649 dari 650 dokumen.
- **Ambang 1,0** menangkap 0,44 salinan persis, yaitu yang kosinus terhitungnya kebetulan dibulatkan sedikit di atas 1. Hash md5 menangkap semuanya.
- **MinHash-LSH lebih ketat pada angka ambang yang sama.** Mengganti 3 dari 60 kata mengubah hingga 9 dari 58 shingle, sehingga edit 5% bermedian Jaccard 0,736, di bawah 0,8, padahal kosinus TF-IDF-nya 0,944.

**Kandidat LSH bukan keputusan.** Peluang sebuah pasangan menjadi kandidat mengikuti kurva S, 1 − (1 − sʳ)ᵇ, dengan s Jaccard sebenarnya, b jumlah ember, dan r baris per ember. Pada validasi lokal, untuk ambang 0,8 datasketch memilih b = 9 dan r = 13: pasangan berjaccard tepat 0,8 hanya berpeluang 0,40 menjadi kandidat, sedangkan pasangan 0,7 masih 0,08. Dari 58 kandidat, 8 di bawah ambang. Tanpa verifikasi, LSH membuang 0,24 boilerplate dan 0,08 edit 5% secara kebetulan; dengan verifikasi keduanya 0. Di korpus ini kandidat kebetulan itu duplikat sungguhan, di korpus lain bisa dokumen unik. Bila edit 5% harus tertangkap, ambangnya yang diturunkan.

Angka "ambang 0,9" bukan sifat near-duplicate, melainkan sifat representasinya: nilai yang sama berarti hal berbeda untuk kosinus TF-IDF dan Jaccard shingle.

**Dedup dan pembagian data latih/uji.** Duplikat yang paling mahal adalah yang menyeberang split: bila kembaran sebuah dokumen uji ada di data latih, skor uji mengukur hafalan, bukan generalisasi. Sel berikut memakai label kelompok korpus di atas (paragraf asal setiap varian) sebagai *oracle*, lalu menghitung dokumen uji yang punya kembaran di data latih dalam tiga skenario: split acak, dedup terpisah di dalam setiap split, dan dekontaminasi, yaitu membuang dokumen latih yang mirip dengan dokumen uji mana pun. Setelah itu, pertanyaan yang sama untuk wikitext-2 sungguhan: adakah potongan validation yang hampir sama dengan potongan train?

In [33]:
# CPU: kebocoran duplikat antar-split (oracle dari label kelompok) dan pemeriksaan train-validation wikitext-2
group = np.array(list(range(N4)) + [i for _, i in lab4])                 # kelompok = paragraf asal


def leaked(train_idx, test_idx):
    tg = set(group[list(train_idx)])
    return sum(1 for j in test_idx if group[j] in tg)


order = list(range(len(C4)))
random.Random(7).shuffle(order)
test_idx, train_idx = order[:len(C4) // 5], order[len(C4) // 5:]
print(f"split acak 80/20: {leaked(train_idx, test_idx)} dari {len(test_idx)} dokumen uji punya kembaran di data latih")

within_tr = [train_idx[k] for k in range(len(train_idx)) if k not in tfidf_dedup([C4[i] for i in train_idx], 0.5)[0]]
within_te = [test_idx[k] for k in range(len(test_idx)) if k not in tfidf_dedup([C4[i] for i in test_idx], 0.5)[0]]
print(f"dedup TF-IDF > 0,5 di dalam setiap split: {leaked(within_tr, within_te)} dari {len(within_te)} masih bocor")

vec = TfidfVectorizer().fit(C4)
cross = cosine_similarity(vec.transform([C4[i] for i in train_idx]), vec.transform([C4[i] for i in test_idx]))
decon_tr = [i for i, s in zip(train_idx, cross.max(axis=1)) if s <= 0.5]
dropped = set(train_idx) - set(decon_tr)
test_groups = set(group[test_idx])
print(f"dekontaminasi (buang dokumen latih dengan kosinus > 0,5 ke dokumen uji mana pun): {leaked(decon_tr, test_idx)} bocor | "
      f"{len(dropped)} dokumen latih dibuang, {sum(1 for i in dropped if group[i] not in test_groups)} di antaranya bukan kembaran")

# wikitext-2: potongan validation (60 kata pertama paragraf) lawan seluruh potongan train, MinHash-LSH + verifikasi Jaccard
train_chunks = P60 if CFG["N_CROSS"] is None else random.Random(3).sample(P60, CFG["N_CROSS"])
V60 = sorted({" ".join(detok(l).split()[:60]) for l in WT["validation"] if is_par(l) and len(detok(l).split()) >= 60})
t0 = time.time()
lsh = MinHashLSH(threshold=0.5, num_perm=128)
train_sh = [set(shingles(p)) for p in train_chunks]
for i, s in enumerate(train_sh):
    lsh.insert(str(i), minhash_of(s))
hits, n_cand = [], 0
for j, p in enumerate(V60):
    s = set(shingles(p))
    cands = lsh.query(minhash_of(s))
    n_cand += len(cands)
    hits += [(jaccard(s, train_sh[int(c)]), j, int(c)) for c in cands if jaccard(s, train_sh[int(c)]) >= 0.5]
hits.sort(reverse=True)
print(f"wikitext-2: {len(V60):,} potongan validation lawan {len(train_chunks):,} potongan train | kandidat LSH {n_cand}, "
      f"terverifikasi Jaccard >= 0,5: {len(hits)} | {time.time() - t0:.1f} s")
for J, j, c in hits[:5]:
    print(f"   J {J:.3f} | val: {V60[j][:90]!r}\n           | train: {train_chunks[c][:90]!r}")
train_pars = {l.strip() for l in WT["train"] if is_par(l)}
val_pars = [l.strip() for l in WT["validation"] if is_par(l)]
print(f"paragraf validation yang persis sama dengan paragraf train: {sum(l in train_pars for l in val_pars)} dari {len(val_pars):,}")

split acak 80/20: 51 dari 130 dokumen uji punya kembaran di data latih
dedup TF-IDF > 0,5 di dalam setiap split: 51 dari 124 masih bocor
dekontaminasi (buang dokumen latih dengan kosinus > 0,5 ke dokumen uji mana pun): 0 bocor | 51 dokumen latih dibuang, 0 di antaranya bukan kembaran
wikitext-2: 1,279 potongan validation lawan 12,320 potongan train | kandidat LSH 5, terverifikasi Jaccard >= 0,5: 4 | 7.4 s
   J 1.000 | val: 'The Boat Race is a side-by-side rowing competition between the University of Oxford (somet'
           | train: 'The Boat Race is a side-by-side rowing competition between the University of Oxford (somet'
   J 0.905 | val: "The first Women's Boat Race took place in 1927, but did not become an annual fixture until"
           | train: "The first Women's Boat Race took place in 1927, but did not become an annual fixture until"
   J 0.762 | val: 'Capel Lligwy has national recognition and statutory protection from alteration as it has b'
           | train: "St Mary's h

Pada validasi lokal (ukuran penuh, CPU), split acak 80/20 membuat 51 dari 130 dokumen uji punya kembaran di data latih. Dedup di dalam setiap split tidak menolong (51 dari 124 masih bocor), karena ia hanya membandingkan dokumen di split yang sama. Dekontaminasi menurunkannya ke 0 dengan membuang 51 dokumen latih, semuanya memang kembaran. Aturannya: dedup **sebelum** membagi data, atau bagi data per kelompok duplikat; bila set uji sudah tetap, misalnya benchmark, buang dokumen **latih** yang mirip dengan set uji.

Split resmi wikitext-2 pun tidak sepenuhnya bersih. Pada validasi lokal, LSH mengusulkan 5 kandidat dari 1.279 potongan validation lawan 12.320 potongan train, dan 4 lolos verifikasi. Paragraf pembuka *The Boat Race is a side-by-side rowing competition ...* muncul di kedua split dengan 60 kata pertama identik (Jaccard 1,0), karena artikel lomba dayung dari tahun berbeda memakai pembuka yang sama. Satu lagi templat tentang dua bangunan berbeda, *Capel Lligwy* dan *St Mary's* (0,762). Sebagai paragraf utuh, hanya 1 dari 1.841 yang persis sama. Kecil, tetapi cukup untuk menunjukkan bahwa split resmi tidak menjamin bebas duplikat, dan parafrase tetap tak terlihat oleh metode berbasis shingle.

Terakhir, kemiripan yang terjadi alami di data nyata dan biaya fungsi buku saat korpus membesar. TF-IDF untuk 12.320 potongan train dikalikan per blok 1.000 baris dalam bentuk sparse, tanpa matriks penuh; lalu fungsi buku, persis seperti tercetak, dibandingkan dengan MinHash-LSH pada n paragraf acak.

In [34]:
# CPU: pasangan mirip alami di wikitext-2 train, lalu skala fungsi buku lawan MinHash-LSH
t0 = time.time()
X = TfidfVectorizer().fit_transform(P60)
pairs = []
for s0 in range(0, X.shape[0], 1000):
    Sb = (X[s0:s0 + 1000] @ X.T).toarray()
    for a in range(Sb.shape[0]):
        Sb[a, :s0 + a + 1] = 0                                           # hanya pasangan (i, j) dengan j > i
    ii, jj = np.where(Sb > 0.5)
    pairs += [(float(Sb[i, j]), s0 + i, int(j)) for i, j in zip(ii, jj)]
pairs.sort(reverse=True)
print(f"{X.shape[0]:,} potongan, {time.time() - t0:.1f} s | pasangan kosinus > 0,5: {len(pairs)}, > 0,8: "
      f"{sum(p[0] > 0.8 for p in pairs)}, > 0,9: {sum(p[0] > 0.9 for p in pairs)}")
for v, i, j in pairs[:4]:
    print(f"   {v:.3f} | {P60[i][:76]!r}\n         | {P60[j][:76]!r}")
rows = []
for n in CFG["SCALE_N"]:
    Cn = random.Random(n).sample(P60, n)
    t1 = time.time()
    kept_n = deduplicate_corpus(Cn)
    t_book = time.time() - t1
    t1 = time.time()
    rem, _, _, _ = lsh_dedup(Cn, 0.8, verify=True)
    rows.append(dict(n=n, fungsi_buku_s=round(t_book, 1), matriks_padat_MB=round(n * n * 8 / 1e6), dibuang_buku=n - len(kept_n),
                     minhash_lsh_s=round(time.time() - t1, 1), dibuang_lsh=len(rem)))
print(pd.DataFrame(rows).to_string(index=False))

12,320 potongan, 6.3 s | pasangan kosinus > 0,5: 581, > 0,8: 22, > 0,9: 8
   0.987 | 'M-5 is maintained by the Michigan Department of Transportation (MDOT) like o'
         | 'M-81 is maintained by the Michigan Department of Transportation (MDOT) like '
   0.982 | 'The county prefects and city and municipal mayors are elected to four-year t'
         | 'The county prefects, city and municipal mayors are elected to four-year term'
   0.968 | "After returning to the United States in January 1919, O'Brien returned to Eu"
         | "After returning to the United States in January 1919, O'Brien revisited Euro"
   0.966 | "After O'Brien's May 1915 commissioning, she sailed off the east coast and in"
         | "After her May 1915 commissioning, O'Brien sailed off the east coast and in t"
   n  fungsi_buku_s  matriks_padat_MB  dibuang_buku  minhash_lsh_s  dibuang_lsh
1000            0.2                 8             0            0.5            0
2000            1.1                32         

- **Data nyata.** Di run historis, sama seperti validasi lokal, 12.320 potongan train menghasilkan 581 pasangan berkosinus di atas 0,5 dan 8 di atas 0,9. Dua jenis pasangan melewati ambang buku: paragraf yang hampir sama (dua paragraf *O'Brien* yang berbeda satu kata), dan **templat tentang entitas berbeda**, seperti jalan raya M-5 dan M-81 yang sama-sama berbunyi *... is maintained by the Michigan Department of Transportation* (0,987). Fungsi buku akan membuang salah satunya, padahal keduanya artikel tentang jalan yang berbeda. Membedakan duplikat dari templat butuh aturan yang tahu entitasnya.
- **Skala.** Matriks kemiripan padat fungsi buku tumbuh kuadratik: 8 MB untuk 1.000 paragraf, 512 MB untuk 8.000, dan 80 GB untuk 100.000. MinHash-LSH menyimpan 128 angka per dokumen dan tumbuh mendekati linear. Pada ukuran tabel, waktu keduanya masih sebanding; batas fungsi buku adalah memorinya. Buku benar bahwa deduplikasi skala besar butuh minhashing, sementara kodenya sendiri adalah versi yang memorinya kuadratik.

## Ringkasan

**Listing buku.** Masalah yang menghentikan skrip bab ini berkisar dari baris `pip` di sel Python dan id `GPT4` sampai berkas input yang tak pernah dibuat. Yang lebih licin justru yang tidak memunculkan error: baris kedua `grammar_score` yang dibuang Python, langdetect dan `df.sample` tanpa seed, `except:` kosong, `.mean()` yang melewati NaN, dan ambang dedup 1,0 yang ditentukan pembulatan. Semuanya diperbaiki di jalur utama; listing aslinya tinggal sebagai kotak singkat.

**Yang diukur laboratorium** (run historis):

- **Perplexity per token bukan termometer kebersihan.** Ia naik lalu turun saat derau ditambah, sedangkan bit per byte naik terus. Teks GPT-2 greedy, teks yang diulang, HTML, dan boilerplate berperplexity *lebih rendah* daripada teks bersih tulisan manusia.
- **Data latih menentukan model, dan "pembersihan" bisa merusak.** Derau di 10%, 30%, dan 100% baris paragraf menambah sekitar +0,06, +0,1, dan +1,2 bit per byte pada teks bersih; prapemrosesan buku menambah +1,7.
- **Pemeriksa aturan punya alarm palsunya sendiri**, dan dedup buku menyimpan versi pertama setiap kelompok mirip, sehingga urutan berkas menentukan yang tersisa.
- **Ambang dedup adalah sifat representasi.** Kandidat LSH perlu diverifikasi, templat tentang entitas berbeda bisa tampak seperti duplikat, dan dedup harus mendahului split latih/uji. Split resmi wikitext-2 pun berbagi beberapa paragraf pembuka (validasi lokal).

**Pegangan praktis.** Simpan teks asli dan pakai versi ternormalisasi hanya sebagai kunci pencocokan; catat setiap baris yang dibuang beserta alasannya; ukur setiap pemeriksa pada data berlabel sebelum dipakai membuang data; laporkan bit per byte dan median di samping perplexity; dan bila memungkinkan, uji keputusan pembersihan dengan melatih model kecil.

Bab berikutnya buku membahas augmentasi data. Pertanyaan bab ini terbawa ke sana: ketika data sintetis ditambahkan, apakah model belajar dari variasinya, atau dari artefak cara data itu dibuat?